In [118]:
from __future__ import annotations

from pathlib import Path
import logging
import json
from typing import Dict, List, Optional, Tuple, Any, Iterable, Sequence

import inspect
import numpy as np
import pandas as pd


import random
from datetime import datetime, timedelta, timezone

# Core Utils 

from utils.paths import get_paths

from utils.file_io import (
    save_data,
    load_json,
)

from utils.logging_setup import (
    configure_logging, 
    log_layer_paths,
)

from utils.pipeline_config_loader import (
    load_pipeline_config,
    set_wandb_dir_from_config,
    export_config_snapshot,
    build_truth_config_block,
)

from utils.truths import (
    make_process_run_id,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record_by_hash,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

# Synthetic Generator
from utils.synthetic.generator.synthetic_profiles import (
    load_rich_profile_csv,
    load_and_merge_rich_profiles,
    load_correlation_pairs_csv,
    load_group_map_csv,
    load_fault_pairings_csv,
)

from utils.synthetic.generator.synthetic_missingness import build_missingness_spec_from_truth_payload

from utils.synthetic.generator.synthetic_generator import (
    SyntheticGenerator, 
    EpisodeSpec,
)

from utils.synthetic.generator.synthetic_postgres_writer import (
    ensure_sequence,
    reserve_next_batch_id,
    reserve_cycle_range,
    reset_sequence,
    reset_synthetic_sequences,
    write_stream_batch,
)

#from utils.synthetic.generator.synthetic_export import export_synthetic_batch_to_parquet

# Postgres

from utils.postgres_util import (
    get_engine_from_env, 
    build_postgres_url, 
    execute_sql, 
    read_sql_dataframe,
)

from utils.layer_postgres_writer import (
    write_layer_dataframe, 
    prepare_layer_dataframe,
)


In [119]:

generation_started_current_datetime = datetime.now()
generation_started_adjusted_time = generation_started_current_datetime - timedelta(hours=4)

generation_started_formatted_datetime = generation_started_adjusted_time.strftime("%m%d%Y_%H%M")

print(f"Starting Synthetic Data Generation at {generation_started_formatted_datetime}")

%store generation_started_current_datetime

Starting Synthetic Data Generation at 04162026_1357
Stored 'generation_started_current_datetime' (datetime)


In [120]:
# --- Notebook params ---
STAGE = "synthetic"
DATASET = "pump"
MODE = "train"
PROFILE = "default"

In [121]:
paths = get_paths()

config_obj = load_pipeline_config(
    config_root=paths.configs,
    stage=STAGE,
    dataset=DATASET,
    mode=MODE,
    profile=PROFILE,
    project_root=paths.root,
)

CONFIG = config_obj.data

SYN_CFG = CONFIG["synthetic"]
PATHS = CONFIG["resolved_paths"]
PIPELINE = CONFIG.get("pipeline", {"execution_mode": "batch", "orchestration_mode": "notebook"})

PIPELINE_MODE = PIPELINE["execution_mode"]
DATASET_NAME = str(CONFIG["dataset"]["name"]).strip().lower()
TRUTH_VERSION = CONFIG["versions"]["truth"]

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])
LOGS_PATH = Path(PATHS["logs_root"])
ARTIFACTS_ROOT = Path(PATHS["artifacts_root"])


TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)



set_wandb_dir_from_config(CONFIG)

print("DATASET_NAME:", DATASET_NAME)
print("TRUTHS_PATH:", TRUTHS_PATH)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)

DATASET_NAME: pump
TRUTHS_PATH: /workspace/artifacts/truths
ARTIFACTS_ROOT: /workspace/artifacts


In [122]:
# Logging Setup

# Create gold log path 
synthetic_log_path = paths.logs / "synthetic_data_generator.log"

# Initial Logger
configure_logging(
    "capstone",
    synthetic_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.synthetic")

# Log load and initiation
logger.info("Synethetic Data Generation starting")

# Log paths loads
log_layer_paths(paths, current_layer="synthetic", logger=logger)


2026-04-16 17:57:23,053 | INFO | capstone.synthetic | Synethetic Data Generation starting
2026-04-16 17:57:23,055 | INFO | capstone.synthetic | Project Root Path Loaded: /workspace
2026-04-16 17:57:23,056 | INFO | capstone.synthetic | Project Logging Path Loaded: /workspace/logs
2026-04-16 17:57:23,059 | INFO | capstone.synthetic | Project Artifacts Path Loaded: /workspace/artifacts
2026-04-16 17:57:23,060 | INFO | capstone.synthetic | Project Notebooks Path Loaded: /workspace/notebooks
2026-04-16 17:57:23,062 | INFO | capstone.synthetic | Project Truths Path Loaded: /workspace/artifacts/truths
2026-04-16 17:57:23,063 | INFO | capstone.synthetic | Project Data Path Loaded: /workspace/data
2026-04-16 17:57:23,064 | INFO | capstone.synthetic | Previous Layer (Gold) Path Loaded: /workspace/data/gold
2026-04-16 17:57:23,065 | INFO | capstone.synthetic | Data Synthetic Path Loaded: /workspace/data/synthetic


In [123]:
def sample_int_range(rng: np.random.Generator, value_or_range, *, low_inclusive: bool = True) -> int:
    """
    Accepts either:
      - int
      - [low, high]
    Returns an int.
    """
    if isinstance(value_or_range, (int, np.integer)):
        return int(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = int(value_or_range[0])
        high = int(value_or_range[1])
        if low_inclusive:
            # numpy integers are low inclusive, high exclusive
            return int(rng.integers(low, high + 1))
        return int(rng.integers(low, high))

    raise TypeError(f"Expected int or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")


def sample_float_range(rng: np.random.Generator, value_or_range) -> float:
    """
    Accepts either:
      - float/int
      - [low, high]
    Returns a float.
    """
    if isinstance(value_or_range, (float, int, np.floating, np.integer)):
        return float(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = float(value_or_range[0])
        high = float(value_or_range[1])
        return float(rng.uniform(low, high))

    raise TypeError(f"Expected float or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")

In [124]:
# Get Latest Truth Hash

def get_latest_truth_hash(
    *,
    truth_index_path: Path,
    layer_name: str,
    dataset_name: str,
) -> str:
    """
    Return the most recent truth_hash for a given layer + dataset from truth_index.jsonl.

    Assumes truth_index.jsonl is append-only and newer entries are later in the file.
    """
    if not truth_index_path.exists():
        raise FileNotFoundError(f"truth_index.jsonl not found: {truth_index_path}")

    dataset_name_norm = str(dataset_name).strip().lower()
    layer_name_norm = str(layer_name).strip().lower()

    latest_record: Optional[Dict[str, Any]] = None

    with truth_index_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            rec_layer = str(rec.get("layer_name", "")).strip().lower()
            rec_dataset = str(rec.get("dataset_name", "")).strip().lower()

            if rec_layer == layer_name_norm and rec_dataset == dataset_name_norm:
                latest_record = rec

    if latest_record is None:
        raise ValueError(
            f"No truth records found for layer='{layer_name}' dataset='{dataset_name}' in {truth_index_path}"
        )

    truth_hash = latest_record.get("truth_hash")
    if truth_hash is None or str(truth_hash).strip() == "":
        raise ValueError("Latest record is missing a usable truth_hash.")

    return str(truth_hash).strip()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


def get_latest_silver_eda_truth_hash(
    *,
    truth_index_path: Path,
    dataset_name: str,
) -> str:
    """
    Convenience wrapper: latest Silver EDA truth hash for a dataset.
    """
    return get_latest_truth_hash(
        truth_index_path=truth_index_path,
        layer_name="silver_eda",
        dataset_name=dataset_name,
    )

In [125]:
# Updated
# --- Notebook params ---
STAGE = SYN_CFG["layer_name"]
DATASET = "pump"
MODE = "train"
PROFILE = "default"

# Parent truth hash from your latest Silver EDA run
SILVER_EDA_TRUTH_HASH = get_latest_silver_eda_truth_hash(truth_index_path=TRUTH_INDEX_PATH, dataset_name=DATASET_NAME,)
print("Latest SILVER_EDA_TRUTH_HASH:", SILVER_EDA_TRUTH_HASH)

# Faults
# Episode overrides (easy test knobs)
PRIMARY_SENSOR = None          # None => first sensor
PRIMARY_FAULT_TYPE = list(SYN_CFG["faults"]["allowed"])



# Episode Settings
NORMAL_BEFORE = list(SYN_CFG["episode"]["normal_before_range"])
BUILDUP = list(SYN_CFG["episode"]["buildup_range"])
FAILURE = list(SYN_CFG["episode"]["failure_range"])
RECOVERY = list(SYN_CFG["episode"]["recovery_range"])
NORMAL_AFTER = list(SYN_CFG["episode"]["normal_after_range"])
MAGNITUDE = list(SYN_CFG["episode"]["magnitude_range"])
BUILDUP_FRACTION = float(SYN_CFG["episode"]["buildup_fraction"])


SYNTH_PROCESS_RUN_ID = make_process_run_id(SYN_CFG["process_run_id_prefix"])

# Outputs
OUTPUT_MODE = SYN_CFG["output_mode"]

# Postgres settings
PG_SCHEMA = str(SYN_CFG["postgres"]["schema"])
TABLE_ARTIFACT_NAME = str(SYN_CFG["postgres"]["table_artifact_name"])

# medallion naming: synthetic_<dataset>_<artifact_name>
ARTIFACT_NAME = "stream"       

# Export
EXPORT_ENABLED = bool(SYN_CFG["export"]["enabled"])
EXPORT_DIRECTORY = str(SYN_CFG["export"]["export_dir_key"])


# ---- Mode switch ----
MODE = str(SYN_CFG["generator"]["mode"])         # "single" | "batch"
TARGET_ROWS = int(SYN_CFG["generator"]["target_rows"])
MAX_EPISODES = int(SYN_CFG["generator"]["max_episodes"])  # safety cap

# ---- policy knobs ----
EPISODE_MAX_ROWS = int(SYN_CFG["generator"]["episode_max_rows"])   # prevents monster episodes; forces multiple episodes in a 10k batch
ALLOW_OVERSHOOT = bool(SYN_CFG["generator"]["allow_overshoot"])    # if True, can overshoot when remaining can't fit minimum core

# ---- failure rarity control (match real dataset frequency) ----
#Forumla to determine rows per failure
#new_rows_per_failure = current_rows_per_failure * (actual_broken_rows / target_broken_rows)

# Real dataset: ~7 failures per 220,320 rows  => ~1 failure per 35,474 rows // 250,000 / 7 = 142_857 -> 35_714.2857
ROWS_PER_FAILURE = int(SYN_CFG["generator"]["rows_per_failure"])  # ~31_474.2857


#PG_SCHEMA = "capstone"
ARTIFACT_NAME = "stream"
TABLE_NAME = f"synthetic_{DATASET_NAME.lower()}_{ARTIFACT_NAME}"

# ---- write mode flags
WRITE_MODE = str(SYN_CFG["generator"]["write_mode"])        # "reset" | "append"
APPEND_MODE = str(SYN_CFG["generator"]["append_mode"])     # "continue" | "renumber"   (only matters if WRITE_MODE="append")

Latest SILVER_EDA_TRUTH_HASH: 9756831dd8f4176e026888892e33e183e809f8a60d8fad378568bfb59d2f313f


----

In [126]:
# --- Synthetic config extracts used by the generator build ---

CAL_CFG = SYN_CFG.get("calibration", {}) or {}
CALIBRATION_ENABLED = bool(CAL_CFG.get("enabled", True))
CALIBRATION_MEAN_WITHIN_K_STD = float(CAL_CFG.get("mean_within_k_std", 1.00))
CALIBRATION_STD_RATIO_BOUNDS = tuple(CAL_CFG.get("std_ratio_bounds", [0.90, 1.35]))

CORRELATION_HOTSPOT_CLUSTERS = SYN_CFG.get("correlation_hotspot_clusters", []) or []
CORRELATION_CLUSTER_DERIVATION = dict(SYN_CFG.get("correlation_cluster_derivation", {}) or {})
FAULT_EXCLUDED_SENSORS = list(SYN_CFG.get("fault_excluded_sensors", ["sensor_15", "sensor_50"]) or [])
CORRELATION_TUNING_CFG = dict(SYN_CFG.get("correlation_tuning", {}) or {})

print("CALIBRATION_ENABLED:", CALIBRATION_ENABLED)
print("CALIBRATION_MEAN_WITHIN_K_STD:", CALIBRATION_MEAN_WITHIN_K_STD)
print("CALIBRATION_STD_RATIO_BOUNDS:", CALIBRATION_STD_RATIO_BOUNDS)
print("CORRELATION_HOTSPOT_CLUSTERS:", CORRELATION_HOTSPOT_CLUSTERS)
print("CORRELATION_CLUSTER_DERIVATION:", CORRELATION_CLUSTER_DERIVATION)
print("FAULT_EXCLUDED_SENSORS:", FAULT_EXCLUDED_SENSORS)
print("CORRELATION_TUNING_CFG:", CORRELATION_TUNING_CFG)

logger.info("CALIBRATION_ENABLED: %s", CALIBRATION_ENABLED)
logger.info("CALIBRATION_MEAN_WITHIN_K_STD: %s", CALIBRATION_MEAN_WITHIN_K_STD)
logger.info("CALIBRATION_STD_RATIO_BOUNDS: %s", CALIBRATION_STD_RATIO_BOUNDS)
logger.info("CORRELATION_HOTSPOT_CLUSTERS: %s", CORRELATION_HOTSPOT_CLUSTERS)
logger.info("CORRELATION_CLUSTER_DERIVATION: %s", CORRELATION_CLUSTER_DERIVATION)
logger.info("FAULT_EXCLUDED_SENSORS: %s", FAULT_EXCLUDED_SENSORS)
logger.info("CORRELATION_TUNING_CFG: %s", CORRELATION_TUNING_CFG)

2026-04-16 17:57:23,252 | INFO | capstone.synthetic | CALIBRATION_ENABLED: True
2026-04-16 17:57:23,255 | INFO | capstone.synthetic | CALIBRATION_MEAN_WITHIN_K_STD: 1.0
2026-04-16 17:57:23,257 | INFO | capstone.synthetic | CALIBRATION_STD_RATIO_BOUNDS: (0.9, 1.35)
2026-04-16 17:57:23,259 | INFO | capstone.synthetic | CORRELATION_HOTSPOT_CLUSTERS: [['sensor_08', 'sensor_09'], ['sensor_14', 'sensor_16'], ['sensor_17', 'sensor_18'], ['sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25'], ['sensor_31', 'sensor_32', 'sensor_33'], ['sensor_34', 'sensor_35', 'sensor_36']]
2026-04-16 17:57:23,261 | INFO | capstone.synthetic | CORRELATION_CLUSTER_DERIVATION: {'enabled': False, 'min_abs_corr': 0.2, 'top_n_pairs': 80, 'min_cluster_size': 2, 'max_cluster_size': 8}
2026-04-16 17:57:23,263 | INFO | capstone.synthetic | FAULT_EXCLUDED_SENSORS: ['sensor_15', 'sensor_50']
2026-04-16 17:57:23,264 | INFO | capstone.synthetic | CORRELATION_TUNING_CFG: {'family_split_ru

CALIBRATION_ENABLED: True
CALIBRATION_MEAN_WITHIN_K_STD: 1.0
CALIBRATION_STD_RATIO_BOUNDS: (0.9, 1.35)
CORRELATION_HOTSPOT_CLUSTERS: [['sensor_08', 'sensor_09'], ['sensor_14', 'sensor_16'], ['sensor_17', 'sensor_18'], ['sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25'], ['sensor_31', 'sensor_32', 'sensor_33'], ['sensor_34', 'sensor_35', 'sensor_36']]
CORRELATION_CLUSTER_DERIVATION: {'enabled': False, 'min_abs_corr': 0.2, 'top_n_pairs': 80, 'min_cluster_size': 2, 'max_cluster_size': 8}
FAULT_EXCLUDED_SENSORS: ['sensor_15', 'sensor_50']
CORRELATION_TUNING_CFG: {'family_split_rules': {'chain_cluster_avg_abs_corr_threshold': 0.76}, 'normal': {'top_pairwise_overlay': {'strength': 0.18, 'top_n': 140, 'min_abs_corr': 0.08, 'smooth_alpha': 0.91}, 'named_cluster_overlay': {'strength': 0.16, 'smooth_alpha': 0.94}, 'anchor_cluster_generation': {'blend': 0.9, 'min_abs_corr': 0.15, 'residual_floor': 0.04, 'smooth_alpha': 0.94}, 'bridge_pair_generation': {'bri

In [127]:
HOTSPOT_CLUSTER_ARTIFACT_PATH = None
HOTSPOT_CLUSTERS_FOR_GENERATOR = CORRELATION_HOTSPOT_CLUSTERS

try:
    hotspot_key = SYN_CFG["silver_eda_artifact_keys"].get("hotspot_clusters_normal")
    if hotspot_key:
        HOTSPOT_CLUSTER_ARTIFACT_PATH = get_artifact_path_from_truth(silver_eda_truth_record, hotspot_key)
except Exception:
    HOTSPOT_CLUSTER_ARTIFACT_PATH = None

if HOTSPOT_CLUSTER_ARTIFACT_PATH:
    hotspot_payload = json.loads(Path(HOTSPOT_CLUSTER_ARTIFACT_PATH).read_text(encoding="utf-8"))

    if isinstance(hotspot_payload, dict) and "clusters" in hotspot_payload:
        HOTSPOT_CLUSTERS_FOR_GENERATOR = hotspot_payload["clusters"]
    elif isinstance(hotspot_payload, list):
        HOTSPOT_CLUSTERS_FOR_GENERATOR = hotspot_payload

print("HOTSPOT_CLUSTER_ARTIFACT_PATH:", HOTSPOT_CLUSTER_ARTIFACT_PATH)
print("HOTSPOT_CLUSTERS_FOR_GENERATOR:", HOTSPOT_CLUSTERS_FOR_GENERATOR)

HOTSPOT_CLUSTER_ARTIFACT_PATH: None
HOTSPOT_CLUSTERS_FOR_GENERATOR: [['sensor_08', 'sensor_09'], ['sensor_14', 'sensor_16'], ['sensor_17', 'sensor_18'], ['sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25'], ['sensor_31', 'sensor_32', 'sensor_33'], ['sensor_34', 'sensor_35', 'sensor_36']]


---

In [128]:
'''
# ---- Mode switch ----
MODE = "batch"         # "single" | "batch"
TARGET_ROWS = 225_000
MAX_EPISODES = 1_000_000  # safety cap

# ---- policy knobs ----
EPISODE_MAX_ROWS = 12_000   # prevents monster episodes; forces multiple episodes in a 10k batch
ALLOW_OVERSHOOT = False    # if True, can overshoot when remaining can't fit minimum core

# ---- failure rarity control (match real dataset frequency) ----
# Real dataset: ~7 failures per 250,000 rows  => ~1 failure per 35,714 rows
ROWS_PER_FAILURE = 32_000  # ~35714.2857



#PG_SCHEMA = "capstone"
#ARTIFACT_NAME = "stream"
TABLE_NAME = f"synthetic_{DATASET_NAME.lower()}_{ARTIFACT_NAME}"

# ---- write mode flags
WRITE_MODE = "reset"       # "reset" | "append"
APPEND_MODE = "renumber"    # "continue" | "renumber"   (only matters if WRITE_MODE="append")
'''


'\n# ---- Mode switch ----\nMODE = "batch"         # "single" | "batch"\nTARGET_ROWS = 225_000\nMAX_EPISODES = 1_000_000  # safety cap\n\n# ---- policy knobs ----\nEPISODE_MAX_ROWS = 12_000   # prevents monster episodes; forces multiple episodes in a 10k batch\nALLOW_OVERSHOOT = False    # if True, can overshoot when remaining can\'t fit minimum core\n\n# ---- failure rarity control (match real dataset frequency) ----\n# Real dataset: ~7 failures per 250,000 rows  => ~1 failure per 35,714 rows\nROWS_PER_FAILURE = 32_000  # ~35714.2857\n\n\n\n#PG_SCHEMA = "capstone"\n#ARTIFACT_NAME = "stream"\nTABLE_NAME = f"synthetic_{DATASET_NAME.lower()}_{ARTIFACT_NAME}"\n\n# ---- write mode flags\nWRITE_MODE = "reset"       # "reset" | "append"\nAPPEND_MODE = "renumber"    # "continue" | "renumber"   (only matters if WRITE_MODE="append")\n'

---

In [129]:

if SILVER_EDA_TRUTH_HASH is None or str(SILVER_EDA_TRUTH_HASH).strip() == "":
    raise ValueError("Set SILVER_EDA_TRUTH_HASH in the parameter cell.")

silver_eda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver_eda",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_EDA_TRUTH_HASH).strip(),
)

PARENT_TRUTH_HASH = get_truth_hash(silver_eda_truth)
SILVER_PREEDA_TRUTH_HASH = get_parent_truth_hash(silver_eda_truth)

silver_preeda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_PREEDA_TRUTH_HASH).strip(),
)

missingness_payload = (
    (silver_preeda_truth.get("runtime_facts", {}) or {})
    .get("missingness_quarantine", {}) or {}
)
missingness_spec = build_missingness_spec_from_truth_payload(missingness_payload)

MISSINGNESS_SENSOR_OVERRIDES = {}

for sensor_name, override_pct in MISSINGNESS_SENSOR_OVERRIDES.items():
    missingness_spec.missingness_pct_all[str(sensor_name)] = float(override_pct)

parent_mode = get_pipeline_mode_from_truth(silver_eda_truth)
if parent_mode:
    PIPELINE_MODE = parent_mode

print("PARENT_TRUTH_HASH:", PARENT_TRUTH_HASH)
print("PIPELINE_MODE:", PIPELINE_MODE)
print("sensor_50 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_50"))
print("sensor_51 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_51"))
print("sensor_15 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_15"))
print("missingness target count:", len(missingness_spec.missingness_pct_all))

logger.info("W&B PARENT_TRUTH_HASH: %s", PARENT_TRUTH_HASH)
logger.info("W&B PIPELINE_MODE: %s", PIPELINE_MODE)
logger.info("sensor_50 target missing pct: %s", missingness_spec.missingness_pct_all.get("sensor_50"))
logger.info("sensor_51 target missing pct: %s", missingness_spec.missingness_pct_all.get("sensor_51"))
logger.info("sensor_15 target missing pct: %s", missingness_spec.missingness_pct_all.get("sensor_15"))
logger.info("missingness target count: %s", len(missingness_spec.missingness_pct_all))


2026-04-16 17:57:23,362 | INFO | capstone.synthetic | W&B PARENT_TRUTH_HASH: 9756831dd8f4176e026888892e33e183e809f8a60d8fad378568bfb59d2f313f
2026-04-16 17:57:23,365 | INFO | capstone.synthetic | W&B PIPELINE_MODE: batch
2026-04-16 17:57:23,367 | INFO | capstone.synthetic | sensor_50 target missing pct: 34.95688090050835
2026-04-16 17:57:23,369 | INFO | capstone.synthetic | sensor_51 target missing pct: 6.982116920842412
2026-04-16 17:57:23,372 | INFO | capstone.synthetic | sensor_15 target missing pct: 100.0
2026-04-16 17:57:23,374 | INFO | capstone.synthetic | missingness target count: 52


PARENT_TRUTH_HASH: 9756831dd8f4176e026888892e33e183e809f8a60d8fad378568bfb59d2f313f
PIPELINE_MODE: batch
sensor_50 target missing pct: 34.95688090050835
sensor_51 target missing pct: 6.982116920842412
sensor_15 target missing pct: 100.0
missingness target count: 52


In [130]:
'''
# Extraction and Merging from Silver PreEDA Truth
# Built to expand on simple version above to resolve a merging issue with pre-eda notebook dropped sensors not adding back into the missing spec and thus not being calculated
# Permenant solution was added to synthetic_missingness.py to resolve issue.
if SILVER_EDA_TRUTH_HASH is None or str(SILVER_EDA_TRUTH_HASH).strip() == "":
    raise ValueError("Set SILVER_EDA_TRUTH_HASH in the parameter cell.")

silver_eda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver_eda",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_EDA_TRUTH_HASH).strip(),
)

PARENT_TRUTH_HASH = get_truth_hash(silver_eda_truth)
SILVER_PREEDA_TRUTH_HASH = get_parent_truth_hash(silver_eda_truth)

silver_preeda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_PREEDA_TRUTH_HASH).strip(),
)

# ------------------------------------------------------------------
# Start from Silver PreEDA missingness quarantine payload
# ------------------------------------------------------------------
missingness_payload = (
    (silver_preeda_truth.get("runtime_facts", {}) or {})
    .get("missingness_quarantine", {}) or {}
)
missingness_payload = dict(missingness_payload)
missingness_spec = build_missingness_spec_from_truth_payload(missingness_payload)


# Pull existing pieces
missingness_pct_all = dict(missingness_payload.get("missingness_pct_all") or {})
missingness_pct_by_state = dict(missingness_payload.get("missingness_pct_by_state") or {})
missingness_state_dependent_flag = dict(
    missingness_payload.get("missingness_state_dependent_flag") or {}
)

# ------------------------------------------------------------------
# Merge dropped-sensor missingness back in directly from the same payload
# ------------------------------------------------------------------
dropped_features = list(missingness_payload.get("dropped_features") or [])
dropped_missing_pct = dict(missingness_payload.get("dropped_missing_pct") or {})

for sensor_name in dropped_features:
    sensor_name = str(sensor_name).strip()
    if sensor_name == "":
        continue

    if sensor_name in dropped_missing_pct:
        missingness_pct_all[sensor_name] = float(dropped_missing_pct[sensor_name])

    if sensor_name not in missingness_state_dependent_flag:
        missingness_state_dependent_flag[sensor_name] = False

# Write merged values back into payload
missingness_payload["missingness_pct_all"] = missingness_pct_all
missingness_payload["missingness_pct_by_state"] = missingness_pct_by_state
missingness_payload["missingness_state_dependent_flag"] = missingness_state_dependent_flag

missingness_spec = build_missingness_spec_from_truth_payload(missingness_payload)

# targeted overrides for known mismatch sensors after Kaggle comparison
MISSINGNESS_SENSOR_OVERRIDES = {}

for sensor_name, override_pct in MISSINGNESS_SENSOR_OVERRIDES.items():
    missingness_spec.missingness_pct_all[str(sensor_name)] = float(override_pct)

parent_mode = get_pipeline_mode_from_truth(silver_eda_truth)
if parent_mode:
    PIPELINE_MODE = parent_mode

print("PARENT_TRUTH_HASH:", PARENT_TRUTH_HASH)
print("PIPELINE_MODE:", PIPELINE_MODE)
print("sensor_50 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_50"))
print("sensor_51 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_51"))
print("sensor_15 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_15"))
print("missingness target count:", len(missingness_spec.missingness_pct_all))

logger.info("W&B PARENT_TRUTH_HASH: %s", PARENT_TRUTH_HASH)
logger.info("W&B PIPELINE_MODE: %s", PIPELINE_MODE)
logger.info("sensor_50 target missing pct: %s", missingness_spec.missingness_pct_all.get("sensor_50"))
logger.info("sensor_51 target missing pct: %s", missingness_spec.missingness_pct_all.get("sensor_51"))
logger.info("sensor_15 target missing pct: %s", missingness_spec.missingness_pct_all.get("sensor_15"))
logger.info("missingness target count: %s", len(missingness_spec.missingness_pct_all))
'''

'\n# Extraction and Merging from Silver PreEDA Truth\n# Built to expand on simple version above to resolve a merging issue with pre-eda notebook dropped sensors not adding back into the missing spec and thus not being calculated\n# Permenant solution was added to synthetic_missingness.py to resolve issue.\nif SILVER_EDA_TRUTH_HASH is None or str(SILVER_EDA_TRUTH_HASH).strip() == "":\n    raise ValueError("Set SILVER_EDA_TRUTH_HASH in the parameter cell.")\n\nsilver_eda_truth = load_truth_record_by_hash(\n    truth_dir=TRUTHS_PATH,\n    layer_name="silver_eda",\n    dataset_name=DATASET_NAME,\n    truth_hash=str(SILVER_EDA_TRUTH_HASH).strip(),\n)\n\nPARENT_TRUTH_HASH = get_truth_hash(silver_eda_truth)\nSILVER_PREEDA_TRUTH_HASH = get_parent_truth_hash(silver_eda_truth)\n\nsilver_preeda_truth = load_truth_record_by_hash(\n    truth_dir=TRUTHS_PATH,\n    layer_name="silver",\n    dataset_name=DATASET_NAME,\n    truth_hash=str(SILVER_PREEDA_TRUTH_HASH).strip(),\n)\n\n# ---------------------

In [131]:
display(missingness_payload)

{'drop_reasons': {'sensor_15': 'all_null',
  'sensor_50': 'missing_rate_above_threshold'},
 'dropped_count': 2,
 'dropped_features': ['sensor_15', 'sensor_50'],
 'dropped_missing_pct': {'sensor_15': 100.0, 'sensor_50': 34.95688090050835},
 'missingness_pct_all': {'sensor_00': 4.633260711692085,
  'sensor_01': 0.16748366013071894,
  'sensor_02': 0.008623819898329702,
  'sensor_03': 0.008623819898329702,
  'sensor_04': 0.008623819898329702,
  'sensor_05': 0.008623819898329702,
  'sensor_06': 2.177741466957153,
  'sensor_07': 2.474128540305011,
  'sensor_08': 2.3179920116194626,
  'sensor_09': 2.0856027596223674,
  'sensor_10': 0.008623819898329702,
  'sensor_11': 0.008623819898329702,
  'sensor_12': 0.008623819898329702,
  'sensor_13': 0.008623819898329702,
  'sensor_14': 0.009531590413943355,
  'sensor_16': 0.014070442992011621,
  'sensor_17': 0.020878721859114015,
  'sensor_18': 0.020878721859114015,
  'sensor_19': 0.007262164124909223,
  'sensor_20': 0.007262164124909223,
  'sensor_21

In [132]:
print("sensor_50 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_50"))
print("sensor_51 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_51"))
print("sensor_15 target missing pct:", missingness_spec.missingness_pct_all.get("sensor_15"))
print("missingness target count:", len(missingness_spec.missingness_pct_all))

sensor_50 target missing pct: 34.95688090050835
sensor_51 target missing pct: 6.982116920842412
sensor_15 target missing pct: 100.0
missingness target count: 52


In [133]:
keys = SYN_CFG["silver_eda_artifact_keys"]

profile_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_normal"])
profile_abnormal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_abnormal"])
profile_recovery_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_recovery"])

corr_pairs_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["corr_pairs_normal"])
group_map_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["group_map_normal"])
fault_pairings_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["fault_pairings_normal"])

print(profile_normal_path)
print(profile_abnormal_path)
print(profile_recovery_path)
print(corr_pairs_normal_path)
print(group_map_normal_path)
print(fault_pairings_normal_path)

logger.info("silver_eda_artifact_keys: %s", keys)


2026-04-16 17:57:23,460 | INFO | capstone.synthetic | silver_eda_artifact_keys: {'profile_normal': 'feature_profile_normal_path', 'profile_abnormal': 'feature_profile_abnormal_path', 'profile_recovery': 'feature_profile_recovery_path', 'corr_pairs_normal': 'sensor_correlation_pairs_normal_path', 'group_map_normal': 'sensor_group_map_normal_path', 'fault_pairings_normal': 'sensor_fault_pairings_normal_path', 'hotspot_clusters_normal': 'sensor_correlation_hotspot_clusters_normal_path'}


/workspace/artifacts/silver_eda/pump/pump__silver_eda__feature_profile__normal.csv
/workspace/artifacts/silver_eda/pump/pump__silver_eda__feature_profile__abnormal.csv
/workspace/artifacts/silver_eda/pump/pump__silver_eda__feature_profile__recovery.csv
/workspace/artifacts/silver_eda/pump/sensor_correlation_pairs_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_group_map_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_fault_pairings_normal.csv


In [134]:
def load_episode_status_counts_json(path: str | Path) -> list[dict]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"episode_status_counts.json not found: {p}")
    data = json.loads(p.read_text())
    if not isinstance(data, list):
        raise ValueError("episode_status_counts.json must be a JSON list of objects")
    return data


EPISODE_STATUS_JSON_PATH = str(ARTIFACTS_ROOT / f"silver_eda/{DATASET_NAME}/{DATASET_NAME}__silver_eda__episode_status_counts.json")

episode_status_rows = load_episode_status_counts_json(EPISODE_STATUS_JSON_PATH)
print("Loaded episodes:", len(episode_status_rows))
print("First row:", episode_status_rows[0] if episode_status_rows else None)

Loaded episodes: 8
First row: {'meta__episode_id': 0, 'normal': 17155, 'normal_percent': 0.9477900552486188, 'abnormal': 1, 'abnormal_percent': 5.524861878453039e-05, 'recovery': 944, 'recovery_percent': 0.052154696132596684, 'episode_total_rows': 18100}


In [135]:
def summarize_episode_status_rows(episode_status_rows: list[dict]) -> dict:
    if not episode_status_rows:
        return {
            "episodes_n": 0,
            "episode_total_median": 0,
            "episode_total_p10": 0,
            "episode_total_p90": 0,
            "failure_rows_default": 0,
            "failure_rows_max": 0,
            "recovery_percent_median": 0.0,
            "recovery_percent_mean": 0.0,
            "recovery_percent_p10": 0.0,
            "recovery_percent_p90": 0.0,
            "normal_percent_median": 0.0,
        }

    df = pd.DataFrame(episode_status_rows).copy()

    zero_series = pd.Series(0, index=df.index, dtype="int64")

    # normalize abnormal -> failure
    abnormal_series = pd.to_numeric(df.get("abnormal", zero_series), errors="coerce").fillna(0).astype(int)
    failure_series = pd.to_numeric(df.get("failure", zero_series), errors="coerce").fillna(0).astype(int)
    df["failure"] = np.maximum(abnormal_series, failure_series)

    df["normal"] = pd.to_numeric(df.get("normal", zero_series), errors="coerce").fillna(0).astype(int)
    df["recovery"] = pd.to_numeric(df.get("recovery", zero_series), errors="coerce").fillna(0).astype(int)

    if "episode_total_rows" not in df.columns:
        df["episode_total_rows"] = df["normal"] + df["failure"] + df["recovery"]

    total = pd.to_numeric(df["episode_total_rows"], errors="coerce").fillna(0).astype(int)
    total_nonzero = total.replace(0, np.nan)

    recovery_pct = (df["recovery"] / total_nonzero).fillna(0.0)
    normal_pct = (df["normal"] / total_nonzero).fillna(0.0)

    return {
        "episodes_n": int(len(df)),
        "episode_total_median": int(np.nanmedian(total)),
        "episode_total_p10": int(np.nanpercentile(total, 10)),
        "episode_total_p90": int(np.nanpercentile(total, 90)),
        "failure_rows_default": int(df["failure"].median()),
        "failure_rows_max": int(df["failure"].max()),
        "recovery_percent_median": float(recovery_pct.median()),
        "recovery_percent_mean": float(recovery_pct.mean()),
        "recovery_percent_p10": float(np.nanpercentile(recovery_pct, 10)),
        "recovery_percent_p90": float(np.nanpercentile(recovery_pct, 90)),
        "normal_percent_median": float(normal_pct.median()),
    }

In [136]:
def choose_episode_phase_lengths(
    rng: np.random.Generator,
    *,
    episode_status_rows: list[dict],
    # knobs
    failure_rows_default: int = 1,
    # how much of "normal" becomes buildup
    buildup_fraction_of_normal: float = 0.10,   # 10% of normal reserved for buildup window
    min_buildup_rows: int = 10,
    min_normal_before_rows: int = 50,
    min_normal_after_rows: int = 50,
    # choose episode total size from the real distribution
    use_real_episode_totals: bool = True,
) -> dict:
    """
    Returns dict with:
      normal_before, buildup, failure, recovery, normal_after, episode_total
    """

    if not episode_status_rows:
        raise ValueError("episode_status_rows is empty")

    # choose a "template" episode from real distribution
    template = episode_status_rows[int(rng.integers(0, len(episode_status_rows)))]
    real_total = int(template.get("episode_total_rows", 0))

    if use_real_episode_totals and real_total > 0:
        episode_total = real_total
    else:
        # fallback: derive from the JSON distribution
        totals = [int(r.get("episode_total_rows", 0)) for r in episode_status_rows if int(r.get("episode_total_rows", 0)) > 0]
        episode_total = int(rng.choice(totals)) if totals else 5000

    # failure (dataset-faithful)
    failure = int(failure_rows_default)

    # recovery: use template recovery_percent if present
    rec_pct = template.get("recovery_percent", None)
    if rec_pct is None:
        rec = int(template.get("recovery", 0))
    else:
        rec = int(round(float(rec_pct) * episode_total))

    # clamp recovery to something valid
    rec = max(0, min(episode_total - failure, rec))

    remaining = episode_total - failure - rec
    if remaining < 0:
        # fallback if episode_total too small
        remaining = 0
        rec = max(0, episode_total - failure)

    # split remaining into normal + buildup
    # buildup is a fraction of remaining normal-ish region
    buildup = int(round(remaining * float(buildup_fraction_of_normal)))
    buildup = max(min_buildup_rows, buildup) if remaining >= min_buildup_rows else min(remaining, min_buildup_rows)

    normal_total = max(0, remaining - buildup)

    # split normal_total into before/after
    nb = max(min_normal_before_rows, int(round(normal_total * 0.50))) if normal_total else 0
    na = normal_total - nb

    # enforce min after if possible
    if na < min_normal_after_rows and normal_total > 0:
        shift = min(min_normal_after_rows - na, max(0, nb - min_normal_before_rows))
        nb -= shift
        na += shift

    # final safety clamp
    nb = max(0, nb)
    na = max(0, na)

    # recompute totals (must sum exactly)
    episode_total_check = nb + buildup + failure + rec + na
    if episode_total_check != episode_total:
        # adjust normal_after to make exact
        na += (episode_total - episode_total_check)
        na = max(0, na)

    return {
        "episode_total": int(nb + buildup + failure + rec + na),
        "normal_before": int(nb),
        "buildup": int(buildup),
        "failure": int(failure),
        "recovery": int(rec),
        "normal_after": int(na),
        "template_episode_id": int(template.get("meta__episode_id", -1)),
        "template_recovery_percent": float(template.get("recovery_percent", float("nan"))) if "recovery_percent" in template else None,
    }

In [137]:
# --- Load profiles (base + dropped) and build generator ---

profile_dir = Path(profile_normal_path).parent

dropped_profile_normal = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__normal.csv"
dropped_profile_abnormal = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__abnormal.csv"
dropped_profile_recovery = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__recovery.csv"

dropped_profile_normal_path = str(dropped_profile_normal) if dropped_profile_normal.exists() else None
dropped_profile_abnormal_path = str(dropped_profile_abnormal) if dropped_profile_abnormal.exists() else None
dropped_profile_recovery_path = str(dropped_profile_recovery) if dropped_profile_recovery.exists() else None

normal_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_normal_path),
    state_scope="normal",
    dropped_profile_csv_path=dropped_profile_normal_path,
)

abnormal_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_abnormal_path),
    state_scope="abnormal",
    dropped_profile_csv_path=dropped_profile_abnormal_path,
)

recovery_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_recovery_path),
    state_scope="recovery",
    dropped_profile_csv_path=dropped_profile_recovery_path,
)

print(
    "Dropped profile files found:",
    dropped_profile_normal.exists(),
    dropped_profile_abnormal.exists(),
    dropped_profile_recovery.exists(),
)

corr_pairs_df = load_correlation_pairs_csv(corr_pairs_normal_path)
group_map_df = load_group_map_csv(group_map_normal_path)
fault_pairings_df = load_fault_pairings_csv(fault_pairings_normal_path)

def build_state_calibration_targets_from_profile_dicts(
    *,
    normal_profiles: dict,
    abnormal_profiles: dict,
    recovery_profiles: dict,
) -> dict:
    def _to_target_dict(profile_dict: dict) -> dict:
        out = {}
        for sensor, prof in profile_dict.items():
            out[str(sensor)] = {
                "mean": float(prof.mean),
                "std": float(prof.std) if pd.notna(prof.std) else 0.0,
            }
        return out

    return {
        "normal": _to_target_dict(normal_profiles),
        "abnormal": _to_target_dict(abnormal_profiles),
        "recovery": _to_target_dict(recovery_profiles),
    }

state_calibration_targets = (
    build_state_calibration_targets_from_profile_dicts(
        normal_profiles=normal_profiles,
        abnormal_profiles=abnormal_profiles,
        recovery_profiles=recovery_profiles,
    )
    if CALIBRATION_ENABLED
    else None
)

generator = SyntheticGenerator(
    normal_profiles=normal_profiles,
    abnormal_profiles=abnormal_profiles,
    recovery_profiles=recovery_profiles,
    correlation_pairs_dataframe=corr_pairs_df,
    group_map_dataframe=group_map_df,
    fault_pairings_dataframe=fault_pairings_df,
    correlation_hotspot_clusters=HOTSPOT_CLUSTERS_FOR_GENERATOR,
    correlation_cluster_derivation=CORRELATION_CLUSTER_DERIVATION,
    fault_excluded_sensors=FAULT_EXCLUDED_SENSORS,
    correlation_tuning=CORRELATION_TUNING_CFG,
    random_seed=int(SYN_CFG["random_seed"]),
    missingness_spec=missingness_spec if "missingness_spec" in globals() else None,
    state_calibration_targets=state_calibration_targets if CALIBRATION_ENABLED else None,
    mean_within_k_std=CALIBRATION_MEAN_WITHIN_K_STD,
    std_ratio_bounds=CALIBRATION_STD_RATIO_BOUNDS,
)

print("Sensors:", len(generator.sensors))
print("First sensors:", generator.sensors[:10])
print("Calibration enabled:", CALIBRATION_ENABLED)
print("Calibration mean_within_k_std:", CALIBRATION_MEAN_WITHIN_K_STD)
print("Calibration std_ratio_bounds:", CALIBRATION_STD_RATIO_BOUNDS)
print("Resolved hotspot clusters:", generator.correlation_hotspot_clusters)
print("Resolved hotspot cluster count:", len(generator.correlation_hotspot_clusters))
print("Fault-eligible sensors:", len([s for s in generator.sensors if s not in generator.fault_excluded_sensors]))
print("Excluded sensors:", sorted(generator.fault_excluded_sensors))
print("Correlation tuning:", generator.correlation_tuning)

logger.info("Generator Run")
logger.info("Generator Sensors Count: %s", len(generator.sensors))
logger.info("Generator Sensors List: %s", generator.sensors)
logger.info("Calibration Enabled: %s", CALIBRATION_ENABLED)
logger.info("Calibration mean_within_k_std: %s", CALIBRATION_MEAN_WITHIN_K_STD)
logger.info("Calibration std_ratio_bounds: %s", CALIBRATION_STD_RATIO_BOUNDS)
logger.info("Resolved hotspot clusters: %s", generator.correlation_hotspot_clusters)
logger.info("Fault excluded sensors: %s", sorted(generator.fault_excluded_sensors))
logger.info("Correlation tuning: %s", generator.correlation_tuning)

2026-04-16 17:57:23,931 | INFO | capstone.synthetic | Generator Run


Dropped profile files found: True True True
Sensors: 52
First sensors: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09']
Calibration enabled: True
Calibration mean_within_k_std: 1.0
Calibration std_ratio_bounds: (0.9, 1.35)
Resolved hotspot clusters: [['sensor_08', 'sensor_09'], ['sensor_14', 'sensor_16'], ['sensor_17', 'sensor_18'], ['sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25'], ['sensor_31', 'sensor_32', 'sensor_33'], ['sensor_34', 'sensor_35', 'sensor_36']]
Resolved hotspot cluster count: 6
Fault-eligible sensors: 50
Excluded sensors: ['sensor_15', 'sensor_50']
Correlation tuning: {'family_split_rules': {'chain_cluster_avg_abs_corr_threshold': 0.76}, 'normal': {'top_pairwise_overlay': {'strength': 0.18, 'top_n': 140, 'min_abs_corr': 0.08, 'smooth_alpha': 0.91}, 'named_cluster_overlay': {'strength': 0.16, 'smooth_alpha': 0.94}, 'anchor_cluster_generation': {'

2026-04-16 17:57:23,933 | INFO | capstone.synthetic | Generator Sensors Count: 52
2026-04-16 17:57:23,934 | INFO | capstone.synthetic | Generator Sensors List: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51']
2026-04-16 17:57:23,935 | INFO | capstone.synthetic | Calibration Enabled: True
2026-04-16 17:57:23,936 | INFO | capstone.synthetic | Calibration mean_within_k_std

In [138]:
# --- Fault eligibility: use the generator-resolved exclusions ---
FAULT_EXCLUDED_SENSORS = set(generator.fault_excluded_sensors)
FAULT_ELIGIBLE_SENSORS = [s for s in generator.sensors if s not in FAULT_EXCLUDED_SENSORS]

print("Fault-eligible sensors:", len(FAULT_ELIGIBLE_SENSORS))
print("Excluded sensors:", sorted(FAULT_EXCLUDED_SENSORS))

logger.info("Fault-eligible sensors: %s", len(FAULT_ELIGIBLE_SENSORS))
logger.info("Excluded sensors: %s", sorted(FAULT_EXCLUDED_SENSORS))

2026-04-16 17:57:23,955 | INFO | capstone.synthetic | Fault-eligible sensors: 50
2026-04-16 17:57:23,958 | INFO | capstone.synthetic | Excluded sensors: ['sensor_15', 'sensor_50']


Fault-eligible sensors: 50
Excluded sensors: ['sensor_15', 'sensor_50']


In [139]:
# --- Load Silver EDA episode composition stats (normal/failure/recovery per episode) ---

episode_stats_path = profile_dir / f"{DATASET_NAME}__silver_eda__episode_status_counts.json"

EPISODE_STATS = None
EPISODE_STATS_PATH_EXISTS = episode_stats_path.exists()

if EPISODE_STATS_PATH_EXISTS:
    raw_episode_stats = json.loads(episode_stats_path.read_text())

    normalized_episode_stats = []
    for row in raw_episode_stats:
        record = dict(row)

        # Silver EDA currently writes abnormal, not failure, in the JSON
        abnormal_rows = int(record.get("abnormal", 0) or 0)
        failure_rows = int(record.get("failure", 0) or 0)

        # Normalize to a shared key so downstream code can rely on "failure"
        record["failure"] = max(failure_rows, abnormal_rows)

        normalized_episode_stats.append(record)

    # keep only episodes that actually contain a failure marker row
    EPISODE_STATS = [
        r for r in normalized_episode_stats
        if int(r.get("failure", 0) or 0) >= 1
    ]

    print("Loaded raw episode stats rows:", len(raw_episode_stats))
    print("Loaded filtered episode stats rows:", len(EPISODE_STATS))
    print("episode_stats_path:", episode_stats_path)

    if len(EPISODE_STATS) == 0:
        raise ValueError(
            "episode_status_counts.json was found, but no usable failure episodes were loaded. "
            "Check whether the JSON uses 'abnormal' or 'failure' keys."
        )
else:
    print("No episode stats json found at:", episode_stats_path)

Loaded raw episode stats rows: 8
Loaded filtered episode stats rows: 7
episode_stats_path: /workspace/artifacts/silver_eda/pump/pump__silver_eda__episode_status_counts.json


In [140]:
EPISODE_STATS_DF = pd.DataFrame(EPISODE_STATS).copy() if EPISODE_STATS else pd.DataFrame()

if not EPISODE_STATS_DF.empty:
    RECOVERY_ROWS_MEDIAN = int(EPISODE_STATS_DF["recovery"].median())
    RECOVERY_ROWS_Q75 = int(EPISODE_STATS_DF["recovery"].quantile(0.75))
    NORMAL_ROWS_MEDIAN = int(EPISODE_STATS_DF["normal"].median())

    print("RECOVERY_ROWS_MEDIAN:", RECOVERY_ROWS_MEDIAN)
    print("RECOVERY_ROWS_Q75:", RECOVERY_ROWS_Q75)
    print("NORMAL_ROWS_MEDIAN:", NORMAL_ROWS_MEDIAN)
else:
    RECOVERY_ROWS_MEDIAN = 0
    RECOVERY_ROWS_Q75 = 0
    NORMAL_ROWS_MEDIAN = 0

RECOVERY_ROWS_MEDIAN: 944
RECOVERY_ROWS_Q75: 2211
NORMAL_ROWS_MEDIAN: 17155


In [141]:
# --- Validate episode stats payload before batch generation ---

if EPISODE_STATS_PATH_EXISTS:
    display(
        pd.DataFrame(EPISODE_STATS)[
            ["meta__episode_id", "normal", "failure", "recovery", "episode_total_rows"]
        ].head(10)
    )

    episode_stats_df = pd.DataFrame(EPISODE_STATS).copy()

    print("Episode stats summary")
    print("rows:", len(episode_stats_df))
    print("failure row min/max:", int(episode_stats_df["failure"].min()), int(episode_stats_df["failure"].max()))
    print("recovery row min/max:", int(episode_stats_df["recovery"].min()), int(episode_stats_df["recovery"].max()))
    print("episode_total_rows min/max:", int(episode_stats_df["episode_total_rows"].min()), int(episode_stats_df["episode_total_rows"].max()))
    print("mean recovery rows:", float(episode_stats_df["recovery"].mean()))
else:
    print("EPISODE_STATS not available; batch generator will use fallback recovery behavior.")

,meta__episode_id,normal,failure,recovery,episode_total_rows
0,0,17155,1,944,18100
1,1,6410,1,3110,9521
2,2,41697,1,1312,43010
3,3,7159,1,605,7765
4,4,49644,1,8390,58035
5,5,4700,1,41,4742
6,6,25267,1,75,25343


Episode stats summary
rows: 7
failure row min/max: 1 1
recovery row min/max: 41 8390
episode_total_rows min/max: 4742 58035
mean recovery rows: 2068.1428571428573


In [142]:

print(inspect.signature(SyntheticGenerator.__init__))

logger.info("Generator Signature Inspection: %s", inspect.signature(SyntheticGenerator.__init__))

2026-04-16 17:57:24,086 | INFO | capstone.synthetic | Generator Signature Inspection: (self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', correlation_hotspot_clusters: 'Optional[List[List[str]]]' = None, correlation_cluster_derivation: 'Optional[Dict[str, object]]' = None, fault_excluded_sensors: 'Optional[List[str]]' = None, correlation_tuning: 'Optional[Dict[str, object]]' = None, random_seed: 'int' = 42, missingness_spec: 'Optional[MissingnessSpec]' = None, state_calibration_targets: 'Optional[StateCalibrationTargets]' = None, mean_within_k_std: 'float' = 1.0, std_ratio_bounds: 'Tuple[float, float]' = (0.5, 1.5)) -> 'None'


(self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', correlation_hotspot_clusters: 'Optional[List[List[str]]]' = None, correlation_cluster_derivation: 'Optional[Dict[str, object]]' = None, fault_excluded_sensors: 'Optional[List[str]]' = None, correlation_tuning: 'Optional[Dict[str, object]]' = None, random_seed: 'int' = 42, missingness_spec: 'Optional[MissingnessSpec]' = None, state_calibration_targets: 'Optional[StateCalibrationTargets]' = None, mean_within_k_std: 'float' = 1.0, std_ratio_bounds: 'Tuple[float, float]' = (0.5, 1.5)) -> 'None'


In [143]:
'''
# ---- Mode switch ----
MODE = "batch"         # "single" | "batch"
TARGET_ROWS = 72_000
MAX_EPISODES = 1_000_000  # safety cap

# ---- policy knobs ----
EPISODE_MAX_ROWS = 3_000   # prevents monster episodes; forces multiple episodes in a 10k batch
ALLOW_OVERSHOOT = False    # if True, can overshoot when remaining can't fit minimum core

# ---- failure rarity control (match real dataset frequency) ----
# Real dataset: ~7 failures per 250,000 rows  => ~1 failure per 35,714 rows
ROWS_PER_FAILURE = 250_000 / 7  # ~35714.2857

'''

def _episode_total_rows(spec: EpisodeSpec) -> int:
    return int(
        spec.normal_before
        + spec.buildup
        + spec.failure
        + spec.recovery
        + spec.normal_after
    )


def _rows_before_broken(spec: EpisodeSpec) -> int:
    """
    Rows that occur before the BROKEN row inside a fault episode.
    """
    return int(spec.normal_before + spec.buildup)


def _rows_after_broken(spec: EpisodeSpec) -> int:
    """
    Rows that occur after the BROKEN row inside a fault episode.
    """
    return int(spec.recovery + spec.normal_after)


def _failure_probability(
    *,
    rows_since_last_broken: int,
    candidate_rows_before_broken: int,
) -> float:
    """
    Stateful probability trigger.

    Instead of using a guessed episode length, use the actual accumulated
    rows since the last broken row plus the rows that would occur before
    the next broken row in the candidate episode.

    Behavior:
    - below ~85% of ROWS_PER_FAILURE: no failure
    - above ~115% of ROWS_PER_FAILURE: guaranteed failure
    - in between: linear probability ramp
    """
    effective_gap = float(rows_since_last_broken) + float(candidate_rows_before_broken)

    start = 0.85 * float(ROWS_PER_FAILURE)
    full = 1.15 * float(ROWS_PER_FAILURE)

    if effective_gap < start:
        return 0.0
    if effective_gap >= full:
        return 1.0

    p = (effective_gap - start) / (full - start)
    return float(min(1.0, max(0.0, p)))


def _is_failure_episode(
    rng: np.random.Generator,
    *,
    rows_since_last_broken: int,
    candidate_rows_before_broken: int,
) -> bool:
    p = _failure_probability(
        rows_since_last_broken=int(rows_since_last_broken),
        candidate_rows_before_broken=int(candidate_rows_before_broken),
    )
    return bool(rng.random() < p)


rng = np.random.default_rng(int(SYN_CFG.get("random_seed", 42)))
ep_cfg = SYN_CFG["episode"]

allowed_faults = (SYN_CFG.get("faults", {}) or {}).get("allowed", [])
if not allowed_faults:
    raise ValueError("SYN_CFG.faults.allowed is empty or missing")

def _safe_scalar_int(x, *, default: int = 0) -> int:
    if x is None:
        return int(default)
    if isinstance(x, (list, tuple)):
        return int(x[0]) if len(x) else int(default)
    return int(x)

def _safe_scalar_float(x, *, default: float = 0.0) -> float:
    if x is None:
        return float(default)
    if isinstance(x, (list, tuple)):
        return float(x[0]) if len(x) else float(default)
    return float(x)

def pick_fault(rng: np.random.Generator) -> str:
    if PRIMARY_FAULT_TYPE is None or str(PRIMARY_FAULT_TYPE).strip() == "":
        return str(rng.choice(allowed_faults))
    if isinstance(PRIMARY_FAULT_TYPE, (list, tuple)):
        return str(rng.choice(list(PRIMARY_FAULT_TYPE)))
    return str(PRIMARY_FAULT_TYPE)

def pick_sensor(rng: np.random.Generator) -> str:
    if PRIMARY_SENSOR is None or str(PRIMARY_SENSOR).strip() == "":
        return str(rng.choice(FAULT_ELIGIBLE_SENSORS))
    chosen = str(PRIMARY_SENSOR).strip()
    if chosen in FAULT_EXCLUDED_SENSORS:
        raise ValueError(f"PRIMARY_SENSOR='{chosen}' is excluded from fault generation.")
    return chosen

def sample_episode_spec(
    rng: np.random.Generator,
    rows_since_last_broken: int = 0,
) -> EpisodeSpec:
    """
    Unconstrained episode sampling (good for MODE='single').

    Uses the same stateful failure trigger as batch mode.
    """
    magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])

    nb = max(0, _safe_scalar_int(sample_int_range(rng, ep_cfg["normal_before_range"]), default=0))
    bu_candidate = max(1, _safe_scalar_int(sample_int_range(rng, ep_cfg["buildup_range"]), default=1))

    is_fail = _is_failure_episode(
        rng,
        rows_since_last_broken=int(rows_since_last_broken),
        candidate_rows_before_broken=int(nb + bu_candidate),
    )

    failure = 1 if is_fail else 0
    buildup = int(bu_candidate) if is_fail else 0
    recovery = (
        max(1, _safe_scalar_int(sample_int_range(rng, ep_cfg["recovery_range"]), default=1))
        if is_fail
        else 0
    )

    return EpisodeSpec(
        primary_sensor=pick_sensor(rng),
        primary_fault_type=pick_fault(rng),
        magnitude=_safe_scalar_float(magnitude, default=1.0),
        normal_before=int(nb),
        buildup=int(buildup),
        failure=int(failure),
        recovery=int(recovery),
        normal_after=_safe_scalar_int(sample_int_range(rng, ep_cfg["normal_after_range"]), default=0),
    )

def sample_episode_spec_fit_remaining(
    rng: np.random.Generator,
    remaining_rows: int,
    *,
    rows_since_last_broken: int,
) -> EpisodeSpec:
    """
    Batch-safe sampling with stateful failure spacing.

    Differences from the old version:
      1) failure probability is based on rows_since_last_broken + candidate rows before broken
      2) no guessed episode length is used for the failure trigger
      3) EPISODE_STATS composition is still used after a failure episode is selected
    """
    remaining_rows = int(max(0, remaining_rows))

    nb_min, nb_max = [int(x) for x in ep_cfg["normal_before_range"]]
    bu_min, bu_max = [int(x) for x in ep_cfg["buildup_range"]]
    rc_min, rc_max = [int(x) for x in ep_cfg["recovery_range"]]
    na_min, na_max = [int(x) for x in ep_cfg["normal_after_range"]]

    # ---------------------------------
    # Build a candidate "rows before broken" estimate
    # ---------------------------------
    candidate_nb = max(0, _safe_scalar_int(sample_int_range(rng, ep_cfg["normal_before_range"]), default=0))

    # Candidate buildup should reflect the notebook's configured buildup fraction
    candidate_bu = max(
        bu_min,
        int(round(candidate_nb * float(BUILDUP_FRACTION))),
    )
    candidate_bu = int(np.clip(candidate_bu, bu_min, bu_max))

    is_fail = _is_failure_episode(
        rng,
        rows_since_last_broken=int(rows_since_last_broken),
        candidate_rows_before_broken=int(candidate_nb + candidate_bu),
    )

    # -------------------------
    # Failure episode path: use real episode composition when available
    # -------------------------
    if is_fail and EPISODE_STATS:
        lengths = choose_episode_phase_lengths(
            rng,
            episode_status_rows=EPISODE_STATS,
            failure_rows_default=1,
            buildup_fraction_of_normal=BUILDUP_FRACTION,
            min_buildup_rows=10,
            min_normal_before_rows=50,
            min_normal_after_rows=50,
            use_real_episode_totals=True,
        )

        nb = int(lengths["normal_before"])
        bu = int(lengths["buildup"])
        fa = int(lengths["failure"])
        rc = int(lengths["recovery"])
        na = int(lengths["normal_after"])

        # respect config rails
        nb = int(np.clip(nb, nb_min, nb_max))
        bu = int(np.clip(bu, bu_min, bu_max))
        fa = int(max(1, fa))
        rc = int(np.clip(rc, rc_min, rc_max))
        na = int(np.clip(na, na_min, na_max))

        # EPISODE_STATS floors
        stats_recovery_floor = max(rc_min, RECOVERY_ROWS_MEDIAN)
        stats_recovery_floor = min(stats_recovery_floor, rc_max)
        rc = max(rc, stats_recovery_floor)

        stats_buildup_floor = max(bu_min, int(round(rc * 0.75)))
        stats_buildup_floor = min(stats_buildup_floor, bu_max)
        bu = max(bu, stats_buildup_floor)

        fixed_core = bu + fa + rc

        # If remaining space cannot fit the failure core, emit a normal-only filler
        if remaining_rows > 0 and remaining_rows < fixed_core:
            magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
            return EpisodeSpec(
                primary_sensor=pick_sensor(rng),
                primary_fault_type=pick_fault(rng),
                magnitude=_safe_scalar_float(magnitude, default=1.0),
                normal_before=int(remaining_rows),
                buildup=0,
                failure=0,
                recovery=0,
                normal_after=0,
            )

        # Cap by EPISODE_MAX_ROWS first, shrinking normals only
        total = nb + bu + fa + rc + na
        if total > EPISODE_MAX_ROWS:
            extra = total - EPISODE_MAX_ROWS
            cut_after = min(extra, na)
            na -= cut_after
            extra -= cut_after

            if extra > 0:
                cut_before = min(extra, nb)
                nb -= cut_before
                extra -= cut_before

        # Then cap by remaining rows, shrinking normals only
        total2 = nb + bu + fa + rc + na
        if remaining_rows > 0 and total2 > remaining_rows:
            extra = total2 - remaining_rows
            cut_after = min(extra, na)
            na -= cut_after
            extra -= cut_after

            if extra > 0:
                cut_before = min(extra, nb)
                nb -= cut_before
                extra -= cut_before

        magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
        return EpisodeSpec(
            primary_sensor=pick_sensor(rng),
            primary_fault_type=pick_fault(rng),
            magnitude=_safe_scalar_float(magnitude, default=1.0),
            normal_before=int(max(0, nb)),
            buildup=int(max(0, bu)),
            failure=int(max(0, fa)),
            recovery=int(max(0, rc)),
            normal_after=int(max(0, na)),
        )

    # -------------------------
    # Non-failure or no EPISODE_STATS fallback
    # -------------------------
    failure = 1 if is_fail else 0

    buildup = (
        max(1, _safe_scalar_int(sample_int_range(rng, ep_cfg["buildup_range"]), default=1))
        if is_fail
        else 0
    )

    recovery = (
        max(1, _safe_scalar_int(sample_int_range(rng, ep_cfg["recovery_range"]), default=1))
        if is_fail
        else 0
    )

    fixed_core = buildup + failure + recovery

    if remaining_rows > 0 and remaining_rows < fixed_core:
        magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
        return EpisodeSpec(
            primary_sensor=pick_sensor(rng),
            primary_fault_type=pick_fault(rng),
            magnitude=_safe_scalar_float(magnitude, default=1.0),
            normal_before=int(remaining_rows),
            buildup=0,
            failure=0,
            recovery=0,
            normal_after=0,
        )

    nb = max(0, _safe_scalar_int(sample_int_range(rng, ep_cfg["normal_before_range"]), default=0))
    na = max(0, _safe_scalar_int(sample_int_range(rng, ep_cfg["normal_after_range"]), default=0))

    total = nb + fixed_core + na
    if total > EPISODE_MAX_ROWS:
        extra = total - EPISODE_MAX_ROWS
        cut_after = min(extra, na)
        na -= cut_after
        extra -= cut_after
        if extra > 0:
            cut_before = min(extra, nb)
            nb -= cut_before
            extra -= cut_before

    total2 = nb + fixed_core + na
    if remaining_rows > 0 and total2 > remaining_rows:
        extra = total2 - remaining_rows
        cut_after = min(extra, na)
        na -= cut_after
        extra -= cut_after
        if extra > 0:
            cut_before = min(extra, nb)
            nb -= cut_before
            extra -= cut_before

    magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
    return EpisodeSpec(
        primary_sensor=pick_sensor(rng),
        primary_fault_type=pick_fault(rng),
        magnitude=_safe_scalar_float(magnitude, default=1.0),
        normal_before=int(nb),
        buildup=int(buildup),
        failure=int(failure),
        recovery=int(recovery),
        normal_after=int(na),
    )

---

In [144]:
chain_threshold = (
    (generator.correlation_tuning.get("family_split_rules", {}) or {})
    .get("chain_cluster_avg_abs_corr_threshold")
)

print("Loaded chain threshold:", chain_threshold)
print("Rows per failure:", ROWS_PER_FAILURE)
print("Recovery range:", RECOVERY_RANGE if "RECOVERY_RANGE" in globals() else SYN_CFG.get("episode", {}).get("recovery_range"))

cluster_debug_rows = []
for cluster in generator.correlation_hotspot_clusters:
    avg_abs_corr = generator._cluster_avg_abs_corr(cluster)
    family_name = generator._classify_cluster_family(cluster)
    cluster_debug_rows.append({
        "cluster": str(cluster),
        "cluster_size": len(cluster),
        "avg_abs_corr": avg_abs_corr,
        "family_name": family_name,
    })

cluster_debug_df = pd.DataFrame(cluster_debug_rows)
display(cluster_debug_df)

Loaded chain threshold: 0.76
Rows per failure: 28000
Recovery range: [1950, 2850]


,cluster,cluster_size,avg_abs_corr,family_name
0,"['sensor_08', 'sensor_09']",2,0.752197,tight_pair
1,"['sensor_14', 'sensor_16']",2,0.990317,tight_pair
2,"['sensor_17', 'sensor_18']",2,0.983553,tight_pair
3,"['sensor_19', 'sensor_20', 'sensor_21', 'senso...",7,0.984404,dense_cluster
4,"['sensor_31', 'sensor_32', 'sensor_33']",3,0.863155,strong_chain_cluster
5,"['sensor_34', 'sensor_35', 'sensor_36']",3,0.785149,strong_chain_cluster


---

In [ ]:
# This cell is the ONLY place that creates/overwrites synthetic_df.

OBSERVABLE_ZSCORE_THRESHOLD = 2.5
OBSERVABLE_MIN_CONSECUTIVE = 3
#ROWS_SINCE_LAST_BROKEN = 0


if MODE == "single":
    episode = sample_episode_spec(rng)

    synthetic_df = generator.generate_episode(
        episode,
        episode_id=0,
        observable_zscore_threshold=OBSERVABLE_ZSCORE_THRESHOLD,
        observable_min_consecutive=OBSERVABLE_MIN_CONSECUTIVE,
    ).copy()

    # optional legacy compatibility column
    synthetic_df["meta__magnitude"] = float(episode.magnitude)

    print("MODE=single")
    print("Episode:", episode)
    print("rows:", len(synthetic_df))

elif MODE == "batch":
    chunks = []
    row_count = 0
    episode_count = 0
    ROWS_SINCE_LAST_BROKEN = 0

    while row_count < TARGET_ROWS and episode_count < MAX_EPISODES:
        remaining = TARGET_ROWS - row_count
        spec = sample_episode_spec_fit_remaining(
            rng,
            remaining_rows=remaining,
            rows_since_last_broken=ROWS_SINCE_LAST_BROKEN,
        )

        df_ep = generator.generate_episode(
            spec,
            episode_id=episode_count,
            observable_zscore_threshold=OBSERVABLE_ZSCORE_THRESHOLD,
            observable_min_consecutive=OBSERVABLE_MIN_CONSECUTIVE,
        ).copy()

        if int(spec.failure) > 0:
            ROWS_SINCE_LAST_BROKEN = _rows_after_broken(spec)
        else:
            ROWS_SINCE_LAST_BROKEN += _episode_total_rows(spec)

        df_ep["meta__magnitude"] = float(spec.magnitude)

        chunks.append(df_ep)
        row_count += len(df_ep)
        episode_count += 1

        if row_count >= TARGET_ROWS:
            break

    synthetic_df = pd.concat(chunks, ignore_index=True)

    # hard trim to exact size (safe because filler episodes are "normal-only" if needed)
    if len(synthetic_df) > TARGET_ROWS:
        synthetic_df = synthetic_df.iloc[:TARGET_ROWS].copy()

    print("MODE=batch")
    print(
        "episodes_used:",
        int(synthetic_df["meta__episode_id"].nunique())
        if "meta__episode_id" in synthetic_df.columns else None
    )
    print(
        "unique primary_fault_type:",
        synthetic_df["meta__primary_fault_type"].nunique()
        if "meta__primary_fault_type" in synthetic_df.columns else None
    )
    print(
        "unique primary_sensor:",
        synthetic_df["meta__primary_sensor"].nunique()
        if "meta__primary_sensor" in synthetic_df.columns else None
    )
    print("rows:", len(synthetic_df))
    display(synthetic_df["stream_state"].value_counts())

else:
    raise ValueError("MODE must be 'single' or 'batch'")

---

In [ ]:
def apply_missingness_percentage_failsafe(
    dataframe: pd.DataFrame,
    *,
    missingness_pct_all: dict[str, float],
    sensor_columns: Optional[Sequence[str]] = None,
    status_column: str = "machine_status",
    protected_statuses: Sequence[str] = ("BROKEN",),
    phase_column: Optional[str] = "phase",
    protected_phases: Sequence[str] = ("abnormal", "failure", "failed", "fault"),
    random_seed: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Add additional missing values to sensor columns until the final missing
    percentage matches the desired target as closely as possible.

    This function only ADDS missingness. It does not attempt to restore values
    if a column is already over the target missing percentage.

    Parameters
    ----------
    dataframe:
        Final generated dataframe.
    missingness_pct_all:
        Dict like {"sensor_00": 0.1, "sensor_50": 34.96, ...}
        where values are percentages from Silver EDA.
    sensor_columns:
        Optional explicit subset of sensor columns to correct.
        If None, uses keys from missingness_pct_all that exist in dataframe.
    status_column:
        Column used to protect rows such as BROKEN from extra deletion.
    protected_statuses:
        Status values that should not be touched.
    phase_column:
        Optional phase column to protect abnormal/failure-style rows too.
    protected_phases:
        Phase values that should not be touched.
    random_seed:
        RNG seed for reproducibility.

    Returns
    -------
    corrected_df, audit_df
    """
    out = dataframe.copy()
    rng = np.random.default_rng(int(random_seed))

    if sensor_columns is None:
        sensor_columns = [
            col for col in missingness_pct_all.keys()
            if col in out.columns
        ]
    else:
        sensor_columns = [col for col in sensor_columns if col in out.columns]

    protected_statuses_set = {str(x).strip().upper() for x in protected_statuses}
    protected_phases_set = {str(x).strip().lower() for x in protected_phases}

    n_rows = int(len(out))
    audit_rows = []

    if n_rows == 0:
        audit_df = pd.DataFrame(
            columns=[
                "sensor",
                "row_count",
                "target_missing_pct",
                "target_missing_n",
                "current_missing_n",
                "additional_missing_needed",
                "eligible_non_missing_n",
                "applied_missing_n",
                "final_missing_n",
                "final_missing_pct",
                "overshot_before_failsafe",
            ]
        )
        return out, audit_df

    # build a protected row mask once
    protected_mask = pd.Series(False, index=out.index)

    if status_column in out.columns:
        status_series = (
            out[status_column]
            .astype("string")
            .str.upper()
            .fillna("")
        )
        protected_mask = protected_mask | status_series.isin(protected_statuses_set)

    if phase_column is not None and phase_column in out.columns:
        phase_series = (
            out[phase_column]
            .astype("string")
            .str.lower()
            .fillna("")
        )
        protected_mask = protected_mask | phase_series.isin(protected_phases_set)

    for sensor in sensor_columns:
        target_pct = float(missingness_pct_all.get(sensor, 0.0))
        target_missing_n = int(round(n_rows * target_pct / 100.0))

        current_missing_mask = out[sensor].isna()
        current_missing_n = int(current_missing_mask.sum())

        additional_needed = max(0, target_missing_n - current_missing_n)
        overshot_before = current_missing_n > target_missing_n

        eligible_mask = (~current_missing_mask) & (~protected_mask)
        eligible_index = out.index[eligible_mask].to_numpy()
        eligible_n = int(len(eligible_index))

        applied_n = 0
        if additional_needed > 0 and eligible_n > 0:
            sample_n = min(additional_needed, eligible_n)
            chosen_index = rng.choice(eligible_index, size=sample_n, replace=False)
            out.loc[chosen_index, sensor] = np.nan
            applied_n = int(sample_n)

        final_missing_n = int(out[sensor].isna().sum())
        final_missing_pct = float(final_missing_n / n_rows * 100.0)

        audit_rows.append(
            {
                "sensor": sensor,
                "row_count": n_rows,
                "target_missing_pct": target_pct,
                "target_missing_n": target_missing_n,
                "current_missing_n": current_missing_n,
                "additional_missing_needed": additional_needed,
                "eligible_non_missing_n": eligible_n,
                "applied_missing_n": applied_n,
                "final_missing_n": final_missing_n,
                "final_missing_pct": final_missing_pct,
                "overshot_before_failsafe": bool(overshot_before),
            }
        )

    audit_df = pd.DataFrame(audit_rows).sort_values(
        ["applied_missing_n", "target_missing_pct"],
        ascending=[False, False],
    ).reset_index(drop=True)

    return out, audit_df

In [ ]:
synthetic_df, missingness_failsafe_audit_df = apply_missingness_percentage_failsafe(
    synthetic_df,
    missingness_pct_all=missingness_spec.missingness_pct_all,
    sensor_columns=[f"sensor_{i:02d}" for i in range(52)],
    status_column="machine_status",
    protected_statuses=("BROKEN",),
    phase_column="phase",
    protected_phases=("abnormal", "failure", "failed", "fault"),
    random_seed=42,
)

display(missingness_failsafe_audit_df.head(20))

----

In [ ]:
print("stream_state counts")
display(synthetic_df["stream_state"].value_counts(dropna=False))

print("phase counts")
display(synthetic_df["phase"].value_counts(dropna=False))

broken_rows = int((synthetic_df["stream_state"] == "broken").sum())
recovering_rows = int((synthetic_df["stream_state"] == "recovering").sum())

print("broken_rows:", broken_rows)
print("recovering_rows:", recovering_rows)

if broken_rows > 0:
    print("recovering_rows_per_broken_row:", recovering_rows / broken_rows)
else:
    print("recovering_rows_per_broken_row: no broken rows")

if EPISODE_STATS:
    episode_stats_df = pd.DataFrame(EPISODE_STATS)
    print("expected mean recovery rows from EPISODE_STATS:", float(episode_stats_df["recovery"].mean()))

In [ ]:
normal_df = synthetic_df.loc[synthetic_df["stream_state"] == "normal"].copy()

variance_check_rows = []
for sensor, prof in normal_profiles.items():
    if sensor not in normal_df.columns:
        continue
    x = pd.to_numeric(normal_df[sensor], errors="coerce").dropna()
    if len(x) < 5:
        continue
    variance_check_rows.append({
        "sensor": sensor,
        "std_actual": float(x.std(ddof=1)),
        "std_profile": float(prof.std),
        "std_ratio": float(x.std(ddof=1) / max(float(prof.std), 1e-6)),
    })

variance_check_df = pd.DataFrame(variance_check_rows).sort_values("std_ratio")
display(variance_check_df.head(15))



In [ ]:
print("rows:", len(synthetic_df))
print(synthetic_df["stream_state"].value_counts().sort_index())
print(synthetic_df["stream_state"].value_counts(normalize=True).sort_index())

print("broken rows:", int((synthetic_df["stream_state"] == "broken").sum()))
print("recovering rows:", int((synthetic_df["stream_state"] == "recovering").sum()))
print("episodes:", synthetic_df["meta__episode_id"].nunique())

print("phase counts:")
print(synthetic_df["phase"].value_counts().sort_index())

allowed_stream_states = {"normal", "broken", "recovering"}
actual_stream_states = set(synthetic_df["stream_state"].dropna().astype(str).unique().tolist())

print("allowed_stream_states:", allowed_stream_states)
print("actual_stream_states:", actual_stream_states)

if not actual_stream_states.issubset(allowed_stream_states):
    raise ValueError(
        f"Unexpected stream_state values found: {sorted(actual_stream_states - allowed_stream_states)}"
    )

In [ ]:
print(synthetic_df["stream_state"].value_counts(normalize=True).sort_index())
print(synthetic_df["stream_state"].value_counts().sort_index())

In [ ]:
print("Columns:", int(len(synthetic_df.columns)))
print("Rows:", int(len(synthetic_df)))

if "sensor_50" in synthetic_df.columns:
    print("Null % sensor_50:", float(synthetic_df["sensor_50"].isna().mean() * 100.0))
else:
    print("Null % sensor_50: NA (column missing)")

if "sensor_15" in synthetic_df.columns:
    print("All-null sensor_15:", bool(synthetic_df["sensor_15"].isna().all()))
else:
    print("All-null sensor_15: NA (column missing)")

# Basic check that we actually have abnormal segments in the output:
if "stream_state" in synthetic_df.columns:
    print("stream_state counts:", synthetic_df["stream_state"].value_counts().to_dict())

In [ ]:
print("machine_status / stream_state counts")
display(synthetic_df["stream_state"].value_counts(dropna=False))

phase_counts = synthetic_df["phase"].value_counts(dropna=False).to_dict()
print("phase_counts:", phase_counts)

broken_rows = int((synthetic_df["stream_state"] == "broken").sum())
recovering_rows = int((synthetic_df["stream_state"] == "recovering").sum())

print("broken_rows:", broken_rows)
print("recovering_rows:", recovering_rows)

if broken_rows > 0:
    print("recovering_rows_per_broken_row:", recovering_rows / broken_rows)
else:
    print("recovering_rows_per_broken_row: no broken rows")

In [ ]:
def table_exists(engine, *, schema: str, table_name: str) -> bool:
    sql = """
    SELECT EXISTS(
      SELECT 1
      FROM information_schema.tables
      WHERE table_schema = :schema AND table_name = :table
    ) AS exists_flag
    """
    df = read_sql_dataframe(engine, sql, params={"schema": schema, "table": table_name})
    return bool(df.loc[0, "exists_flag"])


def drop_table(engine, *, schema: str, table_name: str) -> None:
    execute_sql(engine, f'DROP TABLE IF EXISTS "{schema}"."{table_name}" CASCADE')


def get_max_batch_id(engine, *, schema: str, table_name: str, batch_col: str = "batch_id") -> int:
    fq = f'"{schema}"."{table_name}"'
    df = read_sql_dataframe(engine, f"SELECT COALESCE(MAX({batch_col}), 0) AS max_batch_id FROM {fq}")
    return int(df.loc[0, "max_batch_id"])


def canonicalize_existing_batch_ids(engine, *, schema: str, table_name: str, batch_col: str = "batch_id") -> Dict[str, int]:
    """
    Renumber existing batch ids in-place to 1..N, preserving ascending order.
    Example: [4,5,6,7] -> [1,2,3,4]
    """
    fq = f'"{schema}"."{table_name}"'

    sql = f"""
    WITH b AS (
        SELECT DISTINCT {batch_col} AS old_batch_id
        FROM {fq}
        WHERE {batch_col} IS NOT NULL
    ),
    m AS (
        SELECT old_batch_id,
               DENSE_RANK() OVER (ORDER BY old_batch_id ASC) AS new_batch_id
        FROM b
    )
    UPDATE {fq} t
    SET {batch_col} = m.new_batch_id
    FROM m
    WHERE t.{batch_col} = m.old_batch_id
    """
    execute_sql(engine, sql)

    df = read_sql_dataframe(
        engine,
        f"""
        SELECT
          COUNT(DISTINCT {batch_col}) AS distinct_batches,
          COALESCE(MIN({batch_col}), 0) AS min_batch_id,
          COALESCE(MAX({batch_col}), 0) AS max_batch_id
        FROM {fq}
        """
    )
    return {
        "distinct_batches": int(df.loc[0, "distinct_batches"]),
        "min_batch_id": int(df.loc[0, "min_batch_id"]),
        "max_batch_id": int(df.loc[0, "max_batch_id"]),
    }


    
def choose_batch_id(
    engine,
    *,
    schema: str,
    table_name: str,
    write_mode: str,    # "reset" | "append"
    append_mode: str,   # "continue" | "renumber" (only used if write_mode="append")
    batch_col: str = "batch_id",
) -> int:
    """
    Implements your exact behavior:

    - write_mode="reset":
        drop table, recreate later via writer, next batch_id = 1

    - write_mode="append", append_mode="continue":
        do NOT renumber; next batch_id = MAX(batch_id)+1

    - write_mode="append", append_mode="renumber":
        renumber existing batches to 1..N (in-place), then next batch_id = N+1
        Example existing [4,5,6,7], new insert becomes 5 while table becomes [1,2,3,4,5]
    """
    write_mode = str(write_mode).strip().lower()
    append_mode = str(append_mode).strip().lower()

    if write_mode == "reset":
        # hard reset: drop table; next batch is 1
        drop_table(engine, schema=schema, table_name=table_name)
        return 1

    if write_mode != "append":
        raise ValueError("write_mode must be 'reset' or 'append'")

    exists = table_exists(engine, schema=schema, table_name=table_name)
    if not exists:
        return 1

    if append_mode == "continue":
        return get_max_batch_id(engine, schema=schema, table_name=table_name, batch_col=batch_col) + 1

    if append_mode == "renumber":
        summary = canonicalize_existing_batch_ids(engine, schema=schema, table_name=table_name, batch_col=batch_col)
        return int(summary["max_batch_id"]) + 1

    raise ValueError("append_mode must be 'continue' or 'renumber'")

In [ ]:
engine = get_engine_from_env()

'''
PG_SCHEMA = "capstone"
ARTIFACT_NAME = "stream"
TABLE_NAME = f"synthetic_{DATASET_NAME.lower()}_{ARTIFACT_NAME}"

# ---- write mode flags
WRITE_MODE = "reset"       # "reset" | "append"
APPEND_MODE = "renumber"    # "continue" | "renumber"   (only matters if WRITE_MODE="append")

'''
batch_id = choose_batch_id(
    engine,
    schema=PG_SCHEMA,
    table_name=TABLE_NAME,
    write_mode=WRITE_MODE,
    append_mode=APPEND_MODE,
    batch_col="batch_id",
) 

print("Chosen batch_id:", batch_id)


if str(WRITE_MODE).strip().lower() == "reset":
    ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_batch_id")
    ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id")
    reset_synthetic_sequences(engine, schema=PG_SCHEMA, dataset_name=DATASET_NAME)


# cycles can stay sequence-based for now
ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id")


cycle_start = reserve_cycle_range(
    engine,
    schema=PG_SCHEMA,
    sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id",
    n_rows=len(synthetic_df),
)

table_written = write_stream_batch(
    engine,
    synthetic_df,
    dataset_name=DATASET_NAME,
    schema=PG_SCHEMA,
    artifact_name=ARTIFACT_NAME,
    batch_id=batch_id,
    cycle_start=cycle_start,
)

print("Wrote:", table_written, "batch_id:", batch_id, "cycle_start:", cycle_start)

In [ ]:
import os
{k: os.environ.get(k) for k in ["DB_HOST","DB_PORT","DB_NAME","DB_USER","DB_PASSWORD","POSTGRES_DB","POSTGRES_USER","POSTGRES_PASSWORD"]}

In [ ]:
import importlib.util
print("psycopg:", importlib.util.find_spec("psycopg"))
print("psycopg2:", importlib.util.find_spec("psycopg2"))

In [ ]:
def _as_int(x):
    """
    Coerce x into an int if possible.
    - If x is list/tuple: take the first element (or None if empty)
    - If x is None: return None
    - If x is float/string/int: convert safely
    """
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        if len(x) == 0:
            return None
        x = x[0]
    try:
        return int(x)
    except Exception:
        return None


def _as_float(x):
    """
    Coerce x into a float if possible.
    - If x is list/tuple: take the first element (or None if empty)
    - If x is None: return None
    """
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        if len(x) == 0:
            return None
        x = x[0]
    try:
        return float(x)
    except Exception:
        return None

In [ ]:
# episodes actually created
print("episodes:", synthetic_df["meta__episode_id"].nunique() if "meta__episode_id" in synthetic_df.columns else "NO meta__episode_id")

# how many different primary faults / sensors happened across episodes
if "meta__primary_fault_type" in synthetic_df.columns:
    print("unique primary_fault_type:", synthetic_df["meta__primary_fault_type"].nunique())
    print(synthetic_df[["meta__episode_id","meta__primary_fault_type"]].drop_duplicates().head(10))

if "meta__primary_sensor" in synthetic_df.columns:
    print("unique primary_sensor:", synthetic_df["meta__primary_sensor"].nunique())
    print(synthetic_df[["meta__episode_id","meta__primary_sensor"]].drop_duplicates().head(10))

In [ ]:
current_datetime = datetime.now()
adjusted_time = current_datetime - timedelta(hours=4)

formatted_datetime = adjusted_time.strftime("%m%d%Y_%H%M")

In [ ]:
process_run_id = make_process_run_id(str(SYN_CFG.get("process_run_id_prefix", "synthetic")))

synthetic_truth = initialize_layer_truth(
    truth_version=str(TRUTH_VERSION),
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
    process_run_id=process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=PARENT_TRUTH_HASH,
)

LAYER_NAME = "synthetic"

resolved_config_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
resolved_config_dir.mkdir(parents=True, exist_ok=True)

resolved_config_path = resolved_config_dir / f"{DATASET_NAME}__{LAYER_NAME}__resolved_config.yaml"

# export_config_snapshot requires a destination path
export_config_snapshot(CONFIG, destination=resolved_config_path)

print("Resolved config written to:", resolved_config_path)


synthetic_truth = update_truth_section(
    synthetic_truth,
    "config_snapshot",
    {
        # store the path you just wrote
        "resolved_config_path": str(resolved_config_path),

        # optional: small inline config view (JSON-safe)
        "truth_config_block": build_truth_config_block(CONFIG),

        # keep your synthetic config subset if you want
        "synthetic_cfg": SYN_CFG,
    },
)

synthetic_truth = update_truth_section(
    synthetic_truth,
    "runtime_facts",
    {
        "primary_sensor": (
            getattr(episode, "primary_sensor", None)
            if "episode" in globals()
            else (str(synthetic_df["meta__primary_sensor"].mode(dropna=True).iloc[0]) if "meta__primary_sensor" in synthetic_df.columns else None)
        ),
        "primary_fault_type": (
            getattr(episode, "primary_fault_type", None)
            if "episode" in globals()
            else (str(synthetic_df["meta__primary_fault_type"].mode(dropna=True).iloc[0]) if "meta__primary_fault_type" in synthetic_df.columns else None)
        ),
        "episode": (
            dict(episode.__dict__)
            if ("episode" in globals() and episode is not None and hasattr(episode, "__dict__"))
            else {
                "mode": str(MODE) if "MODE" in globals() else None,
                "episodes_used": int(episode_count) if "episode_count" in globals() else None,
                "target_rows": int(TARGET_ROWS) if "TARGET_ROWS" in globals() else None,
                "meta_episode_id_min": int(synthetic_df["meta__episode_id"].min()) if "meta__episode_id" in synthetic_df.columns else None,
                "meta_episode_id_max": int(synthetic_df["meta__episode_id"].max()) if "meta__episode_id" in synthetic_df.columns else None,
                "meta_primary_sensor_mode": (
                    str(synthetic_df["meta__primary_sensor"].mode(dropna=True).iloc[0])
                    if "meta__primary_sensor" in synthetic_df.columns and not synthetic_df["meta__primary_sensor"].dropna().empty
                    else None
                ),
                "meta_primary_fault_type_mode": (
                    str(synthetic_df["meta__primary_fault_type"].mode(dropna=True).iloc[0])
                    if "meta__primary_fault_type" in synthetic_df.columns and not synthetic_df["meta__primary_fault_type"].dropna().empty
                    else None
                ),
            }
        ),
        "normal_before_range": ep_cfg["normal_before_range"],
        "normal_before_selection": _as_int(NORMAL_BEFORE) if "NORMAL_BEFORE" in globals() else None,
        "buildup_range": ep_cfg["buildup_range"],
        "buildup_selection": _as_int(BUILDUP) if "BUILDUP" in globals() else None,
        "failure_range": ep_cfg["failure_range"],
        "failure_selection": _as_int(FAILURE) if "FAILURE" in globals() else None,
        "recovery_range": ep_cfg["recovery_range"],
        "recovery_selection": _as_int(RECOVERY) if "RECOVERY" in globals() else None,
        "normal_after_range": ep_cfg["normal_after_range"],
        "normal_after_selection": _as_int(NORMAL_AFTER) if "NORMAL_AFTER" in globals() else None,
        "magnitude_range": ep_cfg["magnitude_range"],
        "magnitude_selection": _as_float(MAGNITUDE) if "MAGNITUDE" in globals() else None,
        "row_count": int(len(synthetic_df)),
        "parent_truth_hash": PARENT_TRUTH_HASH,
        "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    },
)

# Optionally save local parquet artifact too (useful for debugging)
synth_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
synth_dir.mkdir(parents=True, exist_ok=True)
suffix = "episode" if ("MODE" in globals() and str(MODE) == "single") else "batch"
out_path = synth_dir / f"{DATASET_NAME}__synthetic__{suffix}.parquet"
save_data(synthetic_df, synth_dir, out_path.name)

artifact_paths_payload = {
    "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    "profile_normal_path": str(profile_normal_path),
    "profile_abnormal_path": str(profile_abnormal_path),
    "profile_recovery_path": str(profile_recovery_path),
    "corr_pairs_normal_path": corr_pairs_normal_path,
    "group_map_normal_path": group_map_normal_path,
    "fault_pairings_normal_path": fault_pairings_normal_path,
    "synthetic_episode_path": str(out_path),
    "postgres_schema": PG_SCHEMA,
    "postgres_table": (
        table_written if "table_written" in globals() else None
    ),  
    "postgres_batch_id": int(batch_id),
    "postgres_cycle_start": int(cycle_start),
}

#export_path = Path(PATHS["data_raw_dir"] / "synthetic") 

if EXPORT_ENABLED:
    artifact_paths_payload["export_batch_parquet_path"] = str(out_path)

synthetic_truth = update_truth_section(synthetic_truth, "artifact_paths", artifact_paths_payload)

meta_columns = sorted(["meta__truth_hash", "meta__parent_truth_hash", "meta__pipeline_mode"])
feature_columns = sorted([column for column in synthetic_df.columns if not str(column).startswith("meta__")])

truth_record = build_truth_record(
    truth_base=synthetic_truth,
    row_count=int(len(synthetic_df)),
    column_count=int(synthetic_df.shape[1] + 3),
    meta_columns=meta_columns,
    feature_columns=feature_columns,
)

synth_truth_hash = truth_record["truth_hash"]

# stamp lineage columns into dataframe (optional)
synthetic_df = stamp_truth_columns(
    synthetic_df,
    truth_hash=synth_truth_hash,
    parent_truth_hash=PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

truth_path = save_truth_record(
    truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
)

append_truth_index(truth_record, truth_index_path=TRUTH_INDEX_PATH)

print("Synthetic truth hash:", synth_truth_hash)
print("Synthetic truth path:", truth_path)
print("Local episode parquet:", out_path)

In [ ]:
from utils.synthetic.generator.synthetic_profiles import load_and_merge_rich_profiles
from utils.synthetic.generator.synthetic_generator import SyntheticGenerator, EpisodeSpec
from utils.synthetic.generator.synthetic_missingness import build_missingness_spec_from_truth_payload
from utils.synthetic.generator.synthetic_export import export_synthetic_batch_to_parquet

print("Imports OK")

In [ ]:
print("Episodes:", synthetic_df["meta__episode_id"].nunique() if "meta__episode_id" in synthetic_df else "NA")
print("Primary fault types:", synthetic_df["meta__primary_fault_type"].value_counts().head(10) if "meta__primary_fault_type" in synthetic_df else "NA")
print("Primary sensors:", synthetic_df["meta__primary_sensor"].value_counts().head(10) if "meta__primary_sensor" in synthetic_df else "NA")

In [ ]:
def verify_schema(
    df,
    *,
    expected_sensors: list[str],
    state_col: str = "stream_state",
    phase_col: str = "phase",
) -> dict:
    cols = set(df.columns)
    exp = set(expected_sensors)

    missing_cols = sorted(exp - cols)
    extra_sensor_cols = sorted([c for c in cols if c.startswith("sensor_") and c not in exp])

    out = {
        "row_count": int(len(df)),
        "missing_sensor_columns": missing_cols,
        "extra_sensor_columns": extra_sensor_cols,
        "state_values": sorted(df[state_col].dropna().astype(str).unique()) if state_col in df.columns else None,
        "phase_values": sorted(df[phase_col].dropna().astype(str).unique()) if phase_col in df.columns else None,
    }
    return out

In [ ]:
def verify_missingness_exact(
    df: pd.DataFrame,
    *,
    target_missing_pct: dict[str, float],
    sensors: list[str],
    tol_missing_rows: int | None = None,
) -> pd.DataFrame:
    n = int(len(df))

    if tol_missing_rows is None:
        tol_missing_rows = max(10, int(round(0.0002 * n)))

    rows = []

    for s in sensors:
        if s not in df.columns:
            continue

        target = float(target_missing_pct.get(s, 0.0))

        # integer expectation consistent with generator logic
        expected_present = int(round(n * (1.0 - target / 100.0)))
        expected_present = max(0, min(n, expected_present))
        expected_missing = n - expected_present

        actual_missing = int(df[s].isna().sum())
        diff_rows = actual_missing - expected_missing

        ok = abs(diff_rows) <= int(tol_missing_rows)

        rows.append({
            "sensor": s,
            "target_pct": target,
            "n_rows": n,
            "expected_missing_rows": expected_missing,
            "actual_missing_rows": actual_missing,
            "diff_missing_rows": diff_rows,
            "ok": ok,
            "actual_pct": float(actual_missing / n * 100.0) if n > 0 else np.nan,
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    return out.sort_values("diff_missing_rows", key=lambda c: c.abs(), ascending=False)

In [ ]:
def verify_profile_bounds(
    df: pd.DataFrame,
    *,
    profile_df: pd.DataFrame,     # rows: sensor,state_scope,p01,p99,mean,std,...
    sensors: list[str],
    state_col: str = "stream_state",
    state_values: list[str] = ["normal","abnormal","recovery"],
    bound_cols: tuple[str, str] = ("p01","p99"),
) -> pd.DataFrame:
    p01_col, p99_col = bound_cols
    prof = profile_df.copy()
    prof["sensor"] = prof["sensor"].astype(str)
    prof["state_scope"] = prof["state_scope"].astype(str)

    rows = []
    for st in state_values:
        df_st = df.loc[df[state_col].astype(str) == st]
        if df_st.empty:
            continue

        for s in sensors:
            if s not in df_st.columns:
                continue

            # profile row
            p = prof.loc[(prof["sensor"] == s) & (prof["state_scope"] == st)]
            if p.empty:
                continue
            p = p.iloc[0]

            series = pd.to_numeric(df_st[s], errors="coerce")
            nonnull = series.dropna()
            if nonnull.empty:
                rows.append({"state": st, "sensor": s, "nonnull_n": 0, "pct_outside_bounds": np.nan})
                continue

            lo = float(p[p01_col])
            hi = float(p[p99_col])

            outside = ((nonnull < lo) | (nonnull > hi)).mean() * 100.0
            rows.append({
                "state": st,
                "sensor": s,
                "nonnull_n": int(nonnull.shape[0]),
                "p01": lo,
                "p99": hi,
                "pct_outside_bounds": float(outside),
                "mean_actual": float(nonnull.mean()),
                "std_actual": float(nonnull.std(ddof=1)) if nonnull.shape[0] > 1 else 0.0,
                "mean_profile": float(p["mean"]) if "mean" in p else np.nan,
                "std_profile": float(p["std"]) if "std" in p else np.nan,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["pct_outside_bounds","nonnull_n"], ascending=[False, True])

In [ ]:
def verify_top_correlations(
    df: pd.DataFrame,
    *,
    corr_pairs_df: pd.DataFrame,
    sensors: list[str],
    state_col: str = "stream_state",
    state_value: str = "normal",
    top_k: int = 25,
) -> pd.DataFrame:
    df_n = df.loc[df[state_col].astype(str) == state_value, sensors].copy()
    df_n = df_n.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    pear = df_n.corr(method="pearson")

    pairs = corr_pairs_df.copy()

    if "pearson_corr" in pairs.columns:
        pairs["expected_abs"] = pairs["pearson_corr"].abs()
        pairs["expected_signed"] = pairs["pearson_corr"]
    elif "correlation" in pairs.columns:
        pairs["expected_abs"] = pairs["correlation"].abs()
        pairs["expected_signed"] = pairs["correlation"]
    else:
        raise ValueError(
            "corr_pairs_df must contain either 'pearson_corr' or 'correlation'."
        )

    pairs = pairs.sort_values("expected_abs", ascending=False).head(top_k).copy()

    rows = []
    for _, r in pairs.iterrows():
        a, b = str(r["sensor_a"]), str(r["sensor_b"])
        if a in pear.columns and b in pear.columns:
            actual = float(pear.loc[a, b])
            rows.append({
                "sensor_a": a,
                "sensor_b": b,
                "expected_abs": float(r["expected_abs"]),
                "actual_abs": float(abs(actual)),
                "actual_signed": actual,
            })

    return pd.DataFrame(rows).sort_values("expected_abs", ascending=False)

In [ ]:
# ---------------------------
# 0) Choose your sensor list
# ---------------------------
SENSORS = list(generator.sensors)  # best source of truth
# For a dataframe view:
# SENSORS = sorted([column for column in synthetic_df.columns if column.startswith("sensor_")])

# ---------------------------
# 1) Schema / state sanity
# ---------------------------
schema_report = verify_schema(
    synthetic_df,
    expected_sensors=SENSORS,
    state_col="stream_state",
    phase_col="phase",
)
print("Schema report:", schema_report)

# ---------------------------
# 2) Missingness sanity
#    (GLOBAL target; easiest / most stable)
# ---------------------------
# Option A: from MissingnessSpec (preferred method)
target_missing_pct = None
if getattr(generator, "missingness_spec", None) is not None:
    target_missing_pct = dict(generator.missingness_spec.missingness_pct_all)

print("Has missingness_spec?:", getattr(generator, "missingness_spec", None) is not None)
print("target_missing_pct size:", 0 if target_missing_pct is None else len(target_missing_pct))

# Option B: from Silver truth
'''
target_missing_pct = (
    (silver_truth.get("runtime_facts", {}) or {})
    .get("missingness_quarantine", {})
    .get("missingness_pct_all")
)

print("target_missing_pct size:", 0 if not target_missing_pct else len(target_missing_pct))
'''

if target_missing_pct is None:
    print("Skipping missingness check: no target_missing_pct available")
else:
    # Force known all-null sensors to 100% missing (matches Silver PreEDA quarantine reality)
    if isinstance(target_missing_pct, dict):
        target_missing_pct = dict(target_missing_pct)  # defensive copy
        target_missing_pct["sensor_15"] = 100.0
        
    missing_report = verify_missingness_exact(
    synthetic_df,
    target_missing_pct=target_missing_pct,
    sensors=generator.sensors,
    tol_missing_rows=None,   # use dynamic tolerance
    )


    display(missing_report.head(30))
    print("Missingness OK?:", bool(missing_report["ok"].all()))

# ---------------------------
# 3) Profile bounds sanity
#    Build one combined profile_df (sensor,state_scope,p01,p99,mean,std,...)
# ---------------------------
def _profiles_dict_to_df(d: dict, state: str) -> pd.DataFrame:
    rows = []
    for sensor, prof in d.items():
        # SensorRichProfile is dataclass-like; __dict__ works
        r = dict(prof.__dict__)
        r["sensor"] = str(sensor)
        r["state_scope"] = str(state)
        rows.append(r)
    return pd.DataFrame(rows)

profile_df = pd.concat(
    [
        _profiles_dict_to_df(normal_profiles, "normal"),
        _profiles_dict_to_df(abnormal_profiles, "abnormal"),
        _profiles_dict_to_df(recovery_profiles, "recovery"),
    ],
    ignore_index=True,
)

print("profile_df columns:", sorted(profile_df.columns.tolist()))
display(profile_df.head(5))



# --- For verification only: map buildup -> abnormal ---
df_check = synthetic_df.copy()
df_check["bounds_state"] = df_check["stream_state"].astype(str)
df_check.loc[df_check["bounds_state"].eq("buildup"), "bounds_state"] = "abnormal"

bounds_check = verify_profile_bounds(
    df_check,
    profile_df=profile_df,
    sensors=SENSORS,
    state_col="bounds_state",
    state_values=["normal", "abnormal", "recovery"],
    bound_cols=("p01", "p99"),
)

display(bounds_check.head(50))


# rule of thumb: if pct_outside_bounds is consistently > ~1-3%,
# your generator clipping or distributions are drifting from profile expectations.

# ---------------------------
# 4) Correlation sanity (normal only)
# ---------------------------
corr_check = verify_top_correlations(
    synthetic_df,
    corr_pairs_df=corr_pairs_df,
    sensors=SENSORS,
    state_col="stream_state",
    state_value="normal",
    top_k=25,
)
display(corr_check)

print("Done.")

In [ ]:
ep0 = synthetic_df.loc[synthetic_df["meta__episode_id"] == 0].copy()
print(ep0["stream_state"].value_counts())

primary = str(ep0["meta__primary_sensor"].dropna().iloc[0])
print("Primary sensor:", primary)

# show some candidate secondaries from your fault_pairings_df
links = fault_pairings_df.loc[fault_pairings_df["sensor_primary"].astype(str) == primary].copy()
display(links.sort_values("fault_coupling_strength", ascending=False).head(10))

In [ ]:
# These should never be selected as primary sensors
if "meta__primary_sensor" in synthetic_df.columns:
    bad = synthetic_df.loc[synthetic_df["meta__primary_sensor"].isin(["sensor_15","sensor_50"])]
    print("Bad primary sensor rows:", len(bad))

# These should be all-null / high-missing as expected
print("sensor_15 missing%:", float(synthetic_df["sensor_15"].isna().mean() * 100.0))
print("sensor_50 missing%:", float(synthetic_df["sensor_50"].isna().mean() * 100.0))

----

In [ ]:
# Dispose Engine
engine.dispose()

---


In [ ]:

def _flatten_config_rows(
    obj: Any,
    prefix: str = "",
) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            key_str = str(key)
            new_prefix = f"{prefix}.{key_str}" if prefix else key_str
            rows.extend(_flatten_config_rows(value, new_prefix))
        return rows

    if isinstance(obj, (list, tuple)):
        rows.append(
            {
                "config_key": prefix,
                "value": json.dumps(list(obj)),
                "value_type": type(obj).__name__,
            }
        )
        return rows

    rows.append(
        {
            "config_key": prefix,
            "value": obj,
            "value_type": type(obj).__name__,
        }
    )
    return rows


def export_config_snapshot_csv(
    *,
    config: Dict[str, Any],
    output_dir: str | Path,
    filename_prefix: str = "synthetic_config_snapshot",
    run_id: Optional[str] = None,
    extra_settings: Optional[Dict[str, Any]] = None,
    timestamp,
) -> tuple[pd.DataFrame, str]:
    """
    Export the currently loaded config plus selected resolved runtime settings
    to a flat CSV for run auditing / upload review.

    Recommended usage:
    - config=CONFIG   -> full loaded config
    - or config=SYN_CFG -> synthetic section only
    - extra_settings=... -> resolved notebook/runtime values you want preserved
    """
    payload: Dict[str, Any] = {"config": config}

    if extra_settings:
        payload["resolved_runtime"] = extra_settings

    rows = _flatten_config_rows(payload)

    df = pd.DataFrame(rows).copy()
    df["exported_at_utc"] = datetime.now(timezone.utc).isoformat()

    if run_id is not None:
        df["run_id"] = str(run_id)

    if not df.empty:
        df["top_level_section"] = (
            df["config_key"]
            .astype(str)
            .str.split(".")
            .str[0]
        )
    else:
        df["top_level_section"] = pd.Series(dtype="string")

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    filename = (
        f"{filename_prefix}__{run_id}__{timestamp}.csv"
        if run_id is not None and str(run_id).strip() != ""
        else f"{filename_prefix}__{timestamp}.csv"
    )

    file_path = output_path / filename
    df.to_csv(file_path, index=False)

    return df, str(file_path)

In [ ]:

paths = get_paths()

EXPORT_SETTINGS_DIR = paths.data_synthetic

In [ ]:
CONFIG_SNAPSHOT_EXTRA = {
    "mode": MODE,
    "target_rows": TARGET_ROWS,
    "max_episodes": MAX_EPISODES,
    "rows_per_failure": ROWS_PER_FAILURE,
    "episode_max_rows": EPISODE_MAX_ROWS,
    "observable_zscore_threshold": OBSERVABLE_ZSCORE_THRESHOLD,
    "observable_min_consecutive": OBSERVABLE_MIN_CONSECUTIVE,
    "silver_eda_truth_hash": SILVER_EDA_TRUTH_HASH,
    "parent_truth_hash": PARENT_TRUTH_HASH,
    "pipeline_mode": PIPELINE_MODE,
    "hotspot_cluster_count": len(HOTSPOT_CLUSTERS_FOR_GENERATOR) if "HOTSPOT_CLUSTERS_FOR_GENERATOR" in globals() else None,
    "fault_excluded_sensors": FAULT_EXCLUDED_SENSORS if "FAULT_EXCLUDED_SENSORS" in globals() else None,
}

config_snapshot_df, config_snapshot_path = export_config_snapshot_csv(
    config=CONFIG,  # or SYN_CFG if you only want the synthetic section
    output_dir=EXPORT_SETTINGS_DIR if "EXPORT_SETTINGS_DIR" in globals() else ".",
    filename_prefix="synthetic_config_snapshot",
    run_id=RUN_ID if "RUN_ID" in globals() else None,
    extra_settings=CONFIG_SNAPSHOT_EXTRA,
    timestamp = formatted_datetime,
)

logger.info("Config snapshot CSV exported: %s", config_snapshot_path)
logger.info("Config snapshot rows: %s", len(config_snapshot_df))

print("Config snapshot CSV:", config_snapshot_path)
display(config_snapshot_df.head(50))



---

In [ ]:



DEFAULT_PRIORITY_SENSORS = [
    "sensor_15",
    "sensor_50",
    "sensor_51",
    "sensor_06",
    "sensor_07",
    "sensor_08",
    "sensor_09",
]

DEFAULT_PRIORITY_PAIRS = [
    ("sensor_08", "sensor_09"),
    ("sensor_14", "sensor_16"),
    ("sensor_17", "sensor_18"),
    ("sensor_20", "sensor_21"),
    ("sensor_22", "sensor_23"),
    ("sensor_25", "sensor_26"),
    ("sensor_31", "sensor_32"),
    ("sensor_32", "sensor_33"),
    ("sensor_34", "sensor_35"),
    ("sensor_35", "sensor_36"),
]

DEFAULT_PRIORITY_CLUSTERS = [
    ["sensor_19", "sensor_20", "sensor_21", "sensor_22", "sensor_23", "sensor_24", "sensor_25"],
    ["sensor_31", "sensor_32", "sensor_33"],
    ["sensor_34", "sensor_35", "sensor_36"],
]

DEFAULT_THRESHOLDS = {
    "state_pct_tolerance": 1.0,            # percentage points
    "broken_count_tolerance": 2,           # rows
    "missing_pct_tolerance": 1.0,          # percentage points
    "priority_pair_abs_gap_pass": 0.20,    # |corr_synth - corr_real|
    "priority_pair_abs_gap_warn": 0.35,
    "cluster_abs_gap_pass": 0.20,
    "cluster_abs_gap_warn": 0.35,
    "median_corr_error_pass": 0.12,
    "median_corr_error_warn": 0.18,
    "p90_corr_error_pass": 0.50,
    "p90_corr_error_warn": 0.70,
    "improvement_epsilon_pct": 0.02,       # 2% relative improvement
}


def _normalize_status_series(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.upper().fillna("UNKNOWN")
    s = s.replace(
        {
            "BROKEN": "BROKEN",
            "ABNORMAL": "BROKEN",
            "FAILURE": "BROKEN",
            "FAILED": "BROKEN",
            "FAULT": "BROKEN",
            "RECOVERING": "RECOVERING",
            "RECOVERY": "RECOVERING",
            "NORMAL": "NORMAL",
        }
    )
    return s.where(s.isin(["NORMAL", "BROKEN", "RECOVERING"]), "NORMAL")


def _sensor_cols(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if str(c).startswith("sensor_")]


def _pct_missing(df: pd.DataFrame, col: str) -> float:
    if col not in df.columns or len(df) == 0:
        return np.nan
    return float(df[col].isna().mean() * 100.0)


def _safe_corr(df: pd.DataFrame, a: str, b: str) -> float:
    if a not in df.columns or b not in df.columns:
        return np.nan
    x = pd.to_numeric(df[a], errors="coerce")
    y = pd.to_numeric(df[b], errors="coerce")
    valid = x.notna() & y.notna()
    if valid.sum() < 3:
        return np.nan
    return float(x[valid].corr(y[valid]))


def _cluster_avg_abs_corr(df: pd.DataFrame, cluster: Sequence[str]) -> float:
    cols = [c for c in cluster if c in df.columns]
    if len(cols) < 2:
        return np.nan

    vals = []
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            c = _safe_corr(df, a, b)
            if np.isfinite(c):
                vals.append(abs(float(c)))

    if not vals:
        return np.nan
    return float(np.mean(vals))


def _overall_corr_errors(
    synth_normal: pd.DataFrame,
    real_normal: pd.DataFrame,
    sensor_cols: Sequence[str],
) -> Dict[str, float]:
    errs: List[float] = []

    cols = [c for c in sensor_cols if c in synth_normal.columns and c in real_normal.columns]
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            cs = _safe_corr(synth_normal, a, b)
            cr = _safe_corr(real_normal, a, b)
            if np.isfinite(cs) and np.isfinite(cr):
                errs.append(abs(float(cs) - float(cr)))

    if not errs:
        return {
            "pair_count": 0,
            "mean_abs_corr_error": np.nan,
            "median_abs_corr_error": np.nan,
            "p90_abs_corr_error": np.nan,
        }

    arr = np.asarray(errs, dtype=float)
    return {
        "pair_count": int(arr.size),
        "mean_abs_corr_error": float(np.mean(arr)),
        "median_abs_corr_error": float(np.median(arr)),
        "p90_abs_corr_error": float(np.percentile(arr, 90)),
    }


def _status_label(value: float, pass_threshold: float, warn_threshold: float) -> str:
    if not np.isfinite(value):
        return "WARN"
    if value <= pass_threshold:
        return "PASS"
    if value <= warn_threshold:
        return "WARN"
    return "FAIL"


def build_synthetic_run_scorecard(
    synthetic_df: pd.DataFrame,
    reference_df: pd.DataFrame,
    *,
    status_col: str = "machine_status",
    priority_sensors: Optional[Sequence[str]] = None,
    priority_pairs: Optional[Sequence[Tuple[str, str]]] = None,
    priority_clusters: Optional[Sequence[Sequence[str]]] = None,
    thresholds: Optional[Dict[str, float]] = None,
    run_id: Optional[str] = None,
) -> Dict[str, pd.DataFrame]:
    """
    Build a scorecard comparing one synthetic run against the reference dataset.
    """
    thresholds = dict(DEFAULT_THRESHOLDS | (thresholds or {}))
    priority_sensors = list(priority_sensors or DEFAULT_PRIORITY_SENSORS)
    priority_pairs = list(priority_pairs or DEFAULT_PRIORITY_PAIRS)
    priority_clusters = [list(c) for c in (priority_clusters or DEFAULT_PRIORITY_CLUSTERS)]

    synth = synthetic_df.copy()
    real = reference_df.copy()

    synth[status_col] = _normalize_status_series(synth[status_col])
    real[status_col] = _normalize_status_series(real[status_col])

    sensor_cols = sorted(set(_sensor_cols(synth)).intersection(_sensor_cols(real)))

    synth_normal = synth.loc[synth[status_col].eq("NORMAL"), sensor_cols].copy()
    real_normal = real.loc[real[status_col].eq("NORMAL"), sensor_cols].copy()

    # -------------------------
    # State summary
    # -------------------------
    synth_counts = synth[status_col].value_counts(dropna=False)
    real_counts = real[status_col].value_counts(dropna=False)

    states = ["NORMAL", "RECOVERING", "BROKEN"]
    state_rows = []
    for state in states:
        synth_n = int(synth_counts.get(state, 0))
        real_n = int(real_counts.get(state, 0))
        synth_pct = float((synth_n / len(synth)) * 100.0) if len(synth) else np.nan
        real_pct = float((real_n / len(real)) * 100.0) if len(real) else np.nan
        pct_gap = abs(synth_pct - real_pct) if np.isfinite(synth_pct) and np.isfinite(real_pct) else np.nan

        status = _status_label(
            pct_gap,
            thresholds["state_pct_tolerance"],
            thresholds["state_pct_tolerance"] * 1.5,
        )

        if state == "BROKEN":
            broken_gap = abs(synth_n - real_n)
            broken_status = _status_label(
                broken_gap,
                thresholds["broken_count_tolerance"],
                thresholds["broken_count_tolerance"] * 2,
            )
        else:
            broken_gap = np.nan
            broken_status = None

        state_rows.append(
            {
                "run_id": run_id,
                "metric_group": "state_mix",
                "state": state,
                "synthetic_n": synth_n,
                "reference_n": real_n,
                "synthetic_pct": synth_pct,
                "reference_pct": real_pct,
                "abs_pct_gap": pct_gap,
                "status": broken_status if state == "BROKEN" else status,
            }
        )

    state_df = pd.DataFrame(state_rows)

    # -------------------------
    # Missingness summary
    # -------------------------
    miss_rows = []
    for sensor in priority_sensors:
        synth_pct = _pct_missing(synth, sensor)
        real_pct = _pct_missing(real, sensor)
        gap = abs(synth_pct - real_pct) if np.isfinite(synth_pct) and np.isfinite(real_pct) else np.nan

        miss_rows.append(
            {
                "run_id": run_id,
                "metric_group": "missingness",
                "sensor": sensor,
                "synthetic_missing_pct": synth_pct,
                "reference_missing_pct": real_pct,
                "abs_gap": gap,
                "status": _status_label(
                    gap,
                    thresholds["missing_pct_tolerance"],
                    thresholds["missing_pct_tolerance"] * 1.5,
                ),
            }
        )

    missingness_df = pd.DataFrame(miss_rows)

    # -------------------------
    # Priority pair correlations
    # -------------------------
    pair_rows = []
    for a, b in priority_pairs:
        cs = _safe_corr(synth_normal, a, b)
        cr = _safe_corr(real_normal, a, b)
        gap = abs(cs - cr) if np.isfinite(cs) and np.isfinite(cr) else np.nan

        pair_rows.append(
            {
                "run_id": run_id,
                "metric_group": "priority_pairs",
                "sensor_a": a,
                "sensor_b": b,
                "synthetic_corr_normal": cs,
                "reference_corr_normal": cr,
                "abs_gap": gap,
                "status": _status_label(
                    gap,
                    thresholds["priority_pair_abs_gap_pass"],
                    thresholds["priority_pair_abs_gap_warn"],
                ),
            }
        )

    pair_df = pd.DataFrame(pair_rows)

    # -------------------------
    # Priority clusters
    # -------------------------
    cluster_rows = []
    for cluster in priority_clusters:
        name = "|".join(cluster)
        cs = _cluster_avg_abs_corr(synth_normal, cluster)
        cr = _cluster_avg_abs_corr(real_normal, cluster)
        gap = abs(cs - cr) if np.isfinite(cs) and np.isfinite(cr) else np.nan

        cluster_rows.append(
            {
                "run_id": run_id,
                "metric_group": "priority_clusters",
                "cluster_name": name,
                "synthetic_avg_abs_corr_normal": cs,
                "reference_avg_abs_corr_normal": cr,
                "abs_gap": gap,
                "status": _status_label(
                    gap,
                    thresholds["cluster_abs_gap_pass"],
                    thresholds["cluster_abs_gap_warn"],
                ),
            }
        )

    cluster_df = pd.DataFrame(cluster_rows)

    # -------------------------
    # Overall corr summary
    # -------------------------
    corr_summary = _overall_corr_errors(synth_normal, real_normal, sensor_cols)
    overall_rows = [
        {
            "run_id": run_id,
            "metric_group": "overall_corr",
            "metric_name": "pair_count",
            "metric_value": corr_summary["pair_count"],
            "status": "INFO",
        },
        {
            "run_id": run_id,
            "metric_group": "overall_corr",
            "metric_name": "mean_abs_corr_error",
            "metric_value": corr_summary["mean_abs_corr_error"],
            "status": "INFO",
        },
        {
            "run_id": run_id,
            "metric_group": "overall_corr",
            "metric_name": "median_abs_corr_error",
            "metric_value": corr_summary["median_abs_corr_error"],
            "status": _status_label(
                corr_summary["median_abs_corr_error"],
                thresholds["median_corr_error_pass"],
                thresholds["median_corr_error_warn"],
            ),
        },
        {
            "run_id": run_id,
            "metric_group": "overall_corr",
            "metric_name": "p90_abs_corr_error",
            "metric_value": corr_summary["p90_abs_corr_error"],
            "status": _status_label(
                corr_summary["p90_abs_corr_error"],
                thresholds["p90_corr_error_pass"],
                thresholds["p90_corr_error_warn"],
            ),
        },
    ]
    overall_df = pd.DataFrame(overall_rows)

    # -------------------------
    # Decision summary
    # -------------------------
    all_statuses = pd.concat(
        [
            state_df[["status"]],
            missingness_df[["status"]],
            pair_df[["status"]],
            cluster_df[["status"]],
            overall_df.loc[overall_df["metric_name"].isin(["median_abs_corr_error", "p90_abs_corr_error"]), ["status"]],
        ],
        ignore_index=True,
    )

    pass_n = int((all_statuses["status"] == "PASS").sum())
    warn_n = int((all_statuses["status"] == "WARN").sum())
    fail_n = int((all_statuses["status"] == "FAIL").sum())

    if fail_n == 0 and warn_n <= 2:
        recommendation = "CANDIDATE_STOP"
    elif fail_n <= 2:
        recommendation = "ONE_MORE_TARGETED_PASS"
    else:
        recommendation = "KEEP_TUNING"

    decision_df = pd.DataFrame(
        [
            {
                "run_id": run_id,
                "pass_count": pass_n,
                "warn_count": warn_n,
                "fail_count": fail_n,
                "recommendation": recommendation,
                "note": (
                    "If two consecutive runs stay flat and recommendation is not KEEP_TUNING, "
                    "draw the line and move on."
                ),
            }
        ]
    )

    return {
        "state_scorecard": state_df,
        "missingness_scorecard": missingness_df,
        "priority_pair_scorecard": pair_df,
        "priority_cluster_scorecard": cluster_df,
        "overall_corr_scorecard": overall_df,
        "decision_scorecard": decision_df,
    }


def export_scorecard_bundle(
    scorecards: Dict[str, pd.DataFrame],
    *,
    output_dir: str | Path,
    run_id: Optional[str] = None,
    prefix: str = "synthetic_scorecard",
) -> Dict[str, str]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    paths: Dict[str, str] = {}
    for name, df in scorecards.items():
        filename = f"{prefix}__{name}"
        if run_id:
            filename += f"__{run_id}"
        filename += ".csv"

        path = output_dir / filename
        df.to_csv(path, index=False)
        paths[name] = str(path)

    return paths


def compare_two_run_decisions(
    current_decision_df: pd.DataFrame,
    previous_decision_df: pd.DataFrame,
) -> pd.DataFrame:
    cur = current_decision_df.iloc[0].to_dict()
    prev = previous_decision_df.iloc[0].to_dict()

    return pd.DataFrame(
        [
            {
                "current_run_id": cur.get("run_id"),
                "previous_run_id": prev.get("run_id"),
                "current_pass_count": cur.get("pass_count"),
                "previous_pass_count": prev.get("pass_count"),
                "current_warn_count": cur.get("warn_count"),
                "previous_warn_count": prev.get("warn_count"),
                "current_fail_count": cur.get("fail_count"),
                "previous_fail_count": prev.get("fail_count"),
                "current_recommendation": cur.get("recommendation"),
                "previous_recommendation": prev.get("recommendation"),
            }
        ]
    )

In [ ]:
# Reference: https://stackoverflow.com/questions/49817409/running-a-jupyter-notebook-from-another-notebook

%run ./synthetic_pipeline_condensed-02_03.ipynb
